# Controlled architecture comparison on three PDE benchmarks

This notebook is a cleaned record of the original training program. Execution outputs and notebook execution metadata were removed; computational cells were preserved to maintain the historical experiment protocol. Dataset access and optional cache paths may require configuration.


In [ ]:

# ============================================================
# 0. Runtime / global protocol
# ============================================================
import os, sys, math, time, json, hashlib, shutil, subprocess, zipfile, copy, gc
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import h5py
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type=="cuda", "A1-R2 formal benchmark requires CUDA."

ROOT=Path(os.getenv("CIDM_BENCH_ROOT", str(Path.cwd())))
OUT=ROOT/"CIDM_Foundation_A1_R2_R1_PUBLICATION_RESULTS"
SRC=OUT/"sources"
FROZEN=OUT/"frozen"
CKPT=OUT/"checkpoints"
FIG=OUT/"figures"
FIGDATA=OUT/"figure_data"
PRED=OUT/"prediction_examples"
for p in [OUT,SRC,FROZEN,CKPT,FIG,FIGDATA,PRED]:
    p.mkdir(parents=True,exist_ok=True)

HISTORY=4
HORIZON=6
TARGET_PARAMS=500_000
PARAM_TOL=0.05

SEEDS=[17,41,73,101,137,181,223,269,313,359]
DEV_SEED=17
LR_GRID=[1e-4,3e-4,1e-3,3e-3]

MAX_UPDATES=int(os.getenv("CIDM_A1_R2_R1_MAX_UPDATES","4000"))
MIN_UPDATES=int(os.getenv("CIDM_A1_R2_R1_MIN_UPDATES","1500"))
EVAL_EVERY=int(os.getenv("CIDM_A1_R2_R1_EVAL_EVERY","100"))
PATIENCE_EVALS=int(os.getenv("CIDM_A1_R2_R1_PATIENCE_EVALS","8"))
REL_MIN_DELTA=float(os.getenv("CIDM_A1_R2_R1_REL_MIN_DELTA","0.001"))

N_TRAIN_TRAJ=int(os.getenv("CIDM_A1_R2_TRAIN_TRAJ","256"))
N_VAL_TRAJ=int(os.getenv("CIDM_A1_R2_VAL_TRAJ","64"))
N_TEST_TRAJ=int(os.getenv("CIDM_A1_R2_TEST_TRAJ","64"))
WINDOWS_PER_TRAJ=int(os.getenv("CIDM_A1_R2_WINDOWS_PER_TRAJ","2"))

BATCH_1D=int(os.getenv("CIDM_A1_R2_BATCH_1D","16"))
BATCH_2D=int(os.getenv("CIDM_A1_R2_BATCH_2D","4"))
WEIGHT_DECAY=1e-4

MODELS=[
    "CIDM-Foundation-v1",
    "CIDM-Complete",
    "PDEBench-FNO",
    "PDEBench-U-Net",
    "NeuralOperator-FNO",
    "Transolver",
]

print("GPU:",torch.cuda.get_device_name(0))
print("Torch:",torch.__version__)
print("Output:",OUT)
print("Convergence protocol:",dict(MAX_UPDATES=MAX_UPDATES,MIN_UPDATES=MIN_UPDATES,EVAL_EVERY=EVAL_EVERY,PATIENCE_EVALS=PATIENCE_EVALS,REL_MIN_DELTA=REL_MIN_DELTA))

REPORT={
    "experiment":"CIDM Foundation A1-R2-R1 Publication-Grade Convergence & Statistical Confirmation",
    "started_at":time.strftime("%Y-%m-%d %H:%M:%S"),
    "protocol":{
        "history":HISTORY,"horizon":HORIZON,
        "target_params":TARGET_PARAMS,"parameter_tolerance":PARAM_TOL,
        "seeds":SEEDS,"dev_seed":DEV_SEED,
        "max_updates":MAX_UPDATES,"min_updates":MIN_UPDATES,
        "eval_every":EVAL_EVERY,"patience_evals":PATIENCE_EVALS,
        "relative_min_delta":REL_MIN_DELTA,
        "train_trajectories":N_TRAIN_TRAJ,
        "val_trajectories":N_VAL_TRAJ,
        "test_trajectories":N_TEST_TRAJ,
        "windows_per_trajectory":WINDOWS_PER_TRAJ,
        "precision":"FP32",
        "optimizer":"AdamW",
        "weight_decay":WEIGHT_DECAY,
    }
}

def run(cmd,cwd=None,timeout=1800):
    p=subprocess.run(
        cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,timeout=timeout
    )
    return p.returncode,p.stdout

def md5_file(path,chunk=8*1024*1024):
    h=hashlib.md5()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""): h.update(b)
    return h.hexdigest()

def sha256_file(path,chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""): h.update(b)
    return h.hexdigest()

def save_json(obj,path):
    Path(path).write_text(
        json.dumps(obj,ensure_ascii=False,indent=2,default=str),
        encoding="utf-8"
    )

def seed_all(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def portable_state_dict(model):
    raw=model.state_dict()
    out=OrderedDict()
    for k,v in raw.items():
        if k.split(".")[-1]=="_metadata":
            continue
        if torch.is_tensor(v):
            out[k]=v.detach().cpu().clone()
        else:
            out[k]=copy.deepcopy(v)
    if hasattr(raw,"_metadata"):
        out._metadata=copy.deepcopy(raw._metadata)
    return out

def load_portable_state_dict(model,sd):
    incompat=model.load_state_dict(sd,strict=False)
    missing=list(incompat.missing_keys)
    unexpected=[k for k in incompat.unexpected_keys if k.split(".")[-1]!="_metadata"]
    if missing or unexpected:
        raise RuntimeError(f"checkpoint mismatch: missing={missing}, unexpected={unexpected}")


In [ ]:

# ============================================================
# 1. Freeze audit:
#    CIDM Foundation Core v1.0 Final
#    + CIDM-Complete archived executable topology
# ============================================================
CIDM_CORE_SOURCE = '"""\nCIDM Foundation Core v1.0\n=========================\n\nFrozen canonical implementation for fair external benchmarking.\n\nThis file contains ONLY the universal core selected after the A1-R1-R3\ncomponent-attribution review:\n\n1. Finite-history predictive information state.\n2. Spatial endpoint encoder.\n3. Temporal-history encoder.\n4. Fusion into a single latent information state.\n5. Bounded local latent residual dynamics.\n6. Bounded positive step size.\n7. Pure latent rollout.\n8. Field decoder.\n\nNot part of the universal core:\n- phase/spectral encoder,\n- skew-symmetric channel rotation,\n- antisymmetric flux,\n- hidden reservoir,\n- causal forcing port,\n- transport branch,\n- cross-scale transfer,\n- BER/SOC,\n- state-sync regularization,\n- Cayley update.\n\nThose are task-specific or training/numerical optionals and must not be silently\nenabled in Foundation Core benchmark runs.\n\nImportant:\nThe numerical caps below are architectural hyperparameters, not physical\nconstants.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n@dataclass(frozen=True)\nclass CIDMFoundationConfig:\n    spatial_dim: int = 2          # 1 or 2\n    channels: int = 1\n    hist_len: int = 4\n    horizon: int = 6\n    latent_dim: int = 32\n    hidden: int = 64\n\n    # Frozen benchmark defaults; architectural hyperparameters, not laws.\n    local_cap: float = 0.10\n    step_cap: float = 0.30\n\n\ndef count_parameters(model: nn.Module) -> int:\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef _conv_nd(dim: int):\n    if dim == 1:\n        return nn.Conv1d\n    if dim == 2:\n        return nn.Conv2d\n    raise ValueError(f"spatial_dim must be 1 or 2, got {dim}")\n\n\nclass ConvBlockND(nn.Module):\n    def __init__(self, dim: int, in_ch: int, out_ch: int, hidden: int):\n        super().__init__()\n        Conv = _conv_nd(dim)\n        self.net = nn.Sequential(\n            Conv(in_ch, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            Conv(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            Conv(hidden, out_ch, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass SpaceEncoder(nn.Module):\n    """Encode the latest observed spatial field X_t."""\n\n    def __init__(self, dim: int, channels: int, latent_dim: int, hidden: int):\n        super().__init__()\n        self.net = ConvBlockND(dim, channels, latent_dim, hidden)\n\n    def forward(self, x_t: torch.Tensor) -> torch.Tensor:\n        return self.net(x_t)\n\n\nclass TimeHistoryEncoder(nn.Module):\n    """Encode finite history H_t = X_{t-L+1:t}."""\n\n    def __init__(\n        self,\n        dim: int,\n        channels: int,\n        hist_len: int,\n        latent_dim: int,\n        hidden: int,\n    ):\n        super().__init__()\n        self.hist_len = hist_len\n        self.channels = channels\n        self.net = ConvBlockND(\n            dim,\n            channels * hist_len,\n            latent_dim,\n            hidden,\n        )\n\n    def forward(self, history: torch.Tensor) -> torch.Tensor:\n        # history: [B, L, C, *S]\n        if history.shape[1] != self.hist_len:\n            raise ValueError(\n                f"Expected history length {self.hist_len}, got {history.shape[1]}"\n            )\n        b, l, c, *spatial = history.shape\n        if c != self.channels:\n            raise ValueError(f"Expected {self.channels} channels, got {c}")\n        x = history.reshape(b, l * c, *spatial)\n        return self.net(x)\n\n\nclass InformationStateFusion(nn.Module):\n    """Fuse spatial endpoint and temporal-history views into Psi_t."""\n\n    def __init__(self, dim: int, latent_dim: int):\n        super().__init__()\n        Conv = _conv_nd(dim)\n        self.net = nn.Sequential(\n            Conv(2 * latent_dim, latent_dim, 1),\n            nn.SiLU(),\n            Conv(latent_dim, latent_dim, 1),\n        )\n\n    def forward(\n        self,\n        z_space: torch.Tensor,\n        z_time: torch.Tensor,\n    ) -> torch.Tensor:\n        return self.net(torch.cat([z_space, z_time], dim=1))\n\n\nclass BoundedLocalResidualDynamics(nn.Module):\n    """\n    Minimal universal transition field.\n\n    The residual branch is bounded twice:\n    - branch responsibility: local_cap * sigmoid(local_logit)\n    - global step: step_cap * sigmoid(step_logit)\n\n    This prevents an unconstrained transition from silently dominating the\n    latent information state.\n    """\n\n    def __init__(\n        self,\n        dim: int,\n        latent_dim: int,\n        hidden: int,\n        local_cap: float,\n        step_cap: float,\n    ):\n        super().__init__()\n        self.local_cap = float(local_cap)\n        self.step_cap = float(step_cap)\n\n        self.local_net = ConvBlockND(\n            dim, latent_dim, latent_dim, hidden\n        )\n        self.local_logit = nn.Parameter(torch.tensor(-2.2))\n        self.step_logit = nn.Parameter(torch.tensor(-1.5))\n\n    def vector_field(self, psi: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        local_scale = self.local_cap * torch.sigmoid(self.local_logit)\n        raw = torch.tanh(self.local_net(psi))\n        delta_local = local_scale * raw\n\n        info = {\n            "local_scale": local_scale,\n            "local_strength": delta_local.abs().mean(),\n        }\n        return delta_local, info\n\n    def forward(\n        self,\n        psi: torch.Tensor,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        delta_local, info = self.vector_field(psi)\n        step = self.step_cap * torch.sigmoid(self.step_logit)\n        psi_next = psi + step * delta_local\n\n        info = {\n            **info,\n            "step_size": step,\n            "delta_strength": (step * delta_local).abs().mean(),\n        }\n        return psi_next, info\n\n\nclass FieldDecoder(nn.Module):\n    def __init__(self, dim: int, latent_dim: int, channels: int, hidden: int):\n        super().__init__()\n        self.net = ConvBlockND(dim, latent_dim, channels, hidden)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.net(psi)\n\n\nclass CIDMFoundationCore(nn.Module):\n    """\n    Closed Information Dynamics Model — Foundation Core v1.0.\n\n    Input:\n        history: [B, L, C, *S]\n\n    Output dict:\n        pred:       [B, K, C, *S]\n        latent0:    [B, D, *S]\n        latent:     [B, D, *S]\n        latent_seq: [B, K, D, *S]\n        diagnostics\n    """\n\n    VERSION = "CIDM-Foundation-Core-v1.0"\n\n    def __init__(self, config: CIDMFoundationConfig):\n        super().__init__()\n        self.config = config\n\n        d = config.spatial_dim\n        D = config.latent_dim\n        H = config.hidden\n\n        self.space_encoder = SpaceEncoder(\n            d, config.channels, D, H\n        )\n        self.time_encoder = TimeHistoryEncoder(\n            d, config.channels, config.hist_len, D, H\n        )\n        self.fusion = InformationStateFusion(d, D)\n\n        self.dynamics = BoundedLocalResidualDynamics(\n            d, D, H,\n            local_cap=config.local_cap,\n            step_cap=config.step_cap,\n        )\n        self.decoder = FieldDecoder(\n            d, D, config.channels, H\n        )\n\n    def encode(self, history: torch.Tensor) -> torch.Tensor:\n        last = history[:, -1]\n        z_space = self.space_encoder(last)\n        z_time = self.time_encoder(history)\n        return self.fusion(z_space, z_time)\n\n    def step(\n        self,\n        psi: torch.Tensor,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        return self.dynamics(psi)\n\n    def decode(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.decoder(psi)\n\n    def forward(\n        self,\n        history: torch.Tensor,\n        horizon: Optional[int] = None,\n        target: Optional[torch.Tensor] = None,\n    ) -> Dict[str, torch.Tensor]:\n        K = int(horizon or self.config.horizon)\n\n        psi0 = self.encode(history)\n        psi = psi0\n\n        preds = []\n        states = []\n        local_strengths = []\n        step_sizes = []\n        delta_strengths = []\n\n        spatial_dims = tuple(range(1, psi.ndim))\n        energy0 = (psi ** 2).mean(dim=spatial_dims)\n        energies = []\n\n        for _ in range(K):\n            psi, info = self.step(psi)\n            y = self.decode(psi)\n\n            preds.append(y)\n            states.append(psi)\n            local_strengths.append(info["local_strength"])\n            step_sizes.append(info["step_size"])\n            delta_strengths.append(info["delta_strength"])\n            energies.append((psi ** 2).mean(dim=spatial_dims))\n\n        pred = torch.stack(preds, dim=1)\n        latent_seq = torch.stack(states, dim=1)\n        energy = torch.stack(energies, dim=1)\n\n        diagnostics: Dict[str, torch.Tensor] = {\n            "local_strength": torch.stack(local_strengths).mean(),\n            "step_size": torch.stack(step_sizes).mean(),\n            "delta_strength": torch.stack(delta_strengths).mean(),\n            "latent_energy_drift": (energy[:, -1] - energy0).abs().mean(),\n        }\n\n        if target is not None:\n            diagnostics["rmse"] = torch.sqrt(F.mse_loss(pred, target))\n            diagnostics["mae"] = F.l1_loss(pred, target)\n\n        return {\n            "pred": pred,\n            "latent0": psi0,\n            "latent": psi,\n            "latent_seq": latent_seq,\n            "diagnostics": diagnostics,\n        }\n\n\ndef build_cidm_foundation_core(\n    *,\n    spatial_dim: int,\n    channels: int,\n    hist_len: int,\n    horizon: int,\n    latent_dim: int,\n    hidden: int,\n    local_cap: float = 0.10,\n    step_cap: float = 0.30,\n) -> CIDMFoundationCore:\n    cfg = CIDMFoundationConfig(\n        spatial_dim=spatial_dim,\n        channels=channels,\n        hist_len=hist_len,\n        horizon=horizon,\n        latent_dim=latent_dim,\n        hidden=hidden,\n        local_cap=local_cap,\n        step_cap=step_cap,\n    )\n    return CIDMFoundationCore(cfg)\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n\n    # 1D smoke\n    m1 = build_cidm_foundation_core(\n        spatial_dim=1,\n        channels=1,\n        hist_len=4,\n        horizon=6,\n        latent_dim=16,\n        hidden=32,\n    )\n    x1 = torch.randn(2, 4, 1, 64)\n    y1 = m1(x1)\n    assert y1["pred"].shape == (2, 6, 1, 64)\n\n    # 2D smoke\n    m2 = build_cidm_foundation_core(\n        spatial_dim=2,\n        channels=2,\n        hist_len=4,\n        horizon=6,\n        latent_dim=16,\n        hidden=32,\n    )\n    x2 = torch.randn(2, 4, 2, 24, 24)\n    y2 = m2(x2)\n    assert y2["pred"].shape == (2, 6, 2, 24, 24)\n\n    print("CIDM Foundation Core v1.0 smoke PASS")\n    print("1D params:", count_parameters(m1))\n    print("2D params:", count_parameters(m2))\n'
CIDM_FREEZE_MANIFEST = '{\n  "name": "CIDM Foundation Core",\n  "version": "v1.0 Final",\n  "freeze_basis": "A1-R1-R3 Core Component Causal Attribution & Freeze",\n  "freeze_gate": "PASS_TO_CORE_FINAL_FREEZE",\n  "default_core": {\n    "encoders": [\n      "SpaceEncoder",\n      "TimeHistoryEncoder"\n    ],\n    "fusion": "InformationStateFusion",\n    "dynamics": "BoundedLocalResidualDynamics",\n    "step": "bounded positive residual step",\n    "decoder": "FieldDecoder",\n    "rollout": "pure latent rollout",\n    "default_training": "prediction / multi-step rollout loss"\n  },\n  "evidence_summary": {\n    "NoSpaceInfo_mean_rmse_effect_pct": 4.895219760746574,\n    "NoTimeInfo_mean_rmse_effect_pct": 5.171493884445902,\n    "NoPhaseInfo_mean_rmse_effect_pct": -2.79857874424354,\n    "NoSkew_mean_rmse_effect_pct": -0.521169227349116,\n    "NoFlux_mean_rmse_effect_pct": -0.9437659673091198,\n    "NoLocal_mean_rmse_effect_pct": 2.397827861124943,\n    "BlackboxCore_mean_rmse_effect_pct": 1.7607338250780888,\n    "DirectPredictor_mean_rmse_effect_pct": 14.230675545900779,\n    "FreeChannelMix_vs_Skew_rmse_change_pct": 3.476865940431788,\n    "FreeFlux_vs_AntisymmetricFlux_rmse_change_pct": -2.5548147465622244,\n    "Sync05_vs_PredOnly_rmse_change_pct": 10.445702293374174,\n    "Sync05_sync_gain_change": 0.5611278432875664\n  },\n  "important_cautions": [\n    "DirectPredictor in A1-R1-R3 was materially smaller than Full, so its 14.23% RMSE gap is supportive rather than a parameter-matched causal estimate.",\n    "Only two seeds were used; several component effects had high seed variance.",\n    "Space/Time encoders show cross-task value but Diffusion-Reaction did not consistently require them.",\n    "The numerical cap values are architectural hyperparameters, not physical constants."\n  ],\n  "optional_components": {\n    "PhaseSpectralEncoder": {\n      "status": "SPECIALIZED OPTIONAL",\n      "reason": "NoPhaseInfo improved mean RMSE by 2.80%; benefits were inconsistent across tasks/seeds."\n    },\n    "SkewSymmetricChannelRotation": {\n      "status": "SPECIALIZED OPTIONAL",\n      "reason": "Removing skew slightly improved mean RMSE; however skew-constrained mixing was better than free channel mixing when a mixing branch was present."\n    },\n    "AntisymmetricFlux": {\n      "status": "SPECIALIZED CONSERVATIVE-EXCHANGE OPTIONAL",\n      "reason": "NoFlux slightly improved mean RMSE and FreeFlux outperformed antisymmetric flux on average."\n    },\n    "HiddenReservoir": {\n      "status": "OPTIONAL",\n      "reason": "Use for unresolved memory/internal closure; not part of the R3 universal core."\n    },\n    "NativeCausalPort": {\n      "status": "OPTIONAL",\n      "reason": "Use only when known causal forcing is available."\n    },\n    "Transport": {\n      "status": "EXPERIMENTAL OPTIONAL",\n      "reason": "Use for displacement/coherent-structure dominated systems."\n    },\n    "CrossScaleTransfer": {\n      "status": "EXPERIMENTAL OPTIONAL",\n      "reason": "Use for tasks with explicit cross-scale transfer."\n    },\n    "StateSyncRegularizer": {\n      "status": "TRAINING OPTIONAL",\n      "reason": "Improved sync metric but increased mean RMSE by 10.45% in R3."\n    },\n    "CayleySkewUpdate": {\n      "status": "NUMERICAL OPTIONAL",\n      "reason": "Strictly orthogonal discretization if a skew branch is deliberately enabled."\n    },\n    "BER_SOC": {\n      "status": "SPECIALIZED EXTENSION FAMILY",\n      "reason": "Keep outside Foundation Core; appropriate for open/physical closure studies."\n    }\n  }\n}'
CIDM_COMPLETE_ARCHIVE_SOURCE = '"""\nCIDM architecture archive\n=========================\n\nReference PyTorch implementation for the finalized model family:\n\n1. CIDM / Closed Information Dynamics Model\n   - General-purpose closed information dynamics backbone.\n   - Corresponds conceptually to the previous experimental `sticm_v3_full`.\n\n2. CIDM-BER / CIDM with Boundary Energy Reservoir\n   - Physics-closure enhanced variant.\n   - Adds a latent Spectral Operator Core and a Boundary Energy Reservoir (BER)\n     residual closure layer.\n   - Corresponds conceptually to the previous experimental `sticm_soc_ber`.\n\nThis file is intended for architecture archiving and future experiment reuse. It\ncontains only model definitions and lightweight diagnostics; training/evaluation\nloops are intentionally kept outside this file.\n\nExpected input format\n---------------------\n`history`: Tensor[B, L, C, H, W]\n`port`: optional Tensor[B, C_port, H, W] or Tensor[B, horizon, C_port, H, W]\n`target`: optional Tensor[B, horizon, C, H, W] for diagnostics/loss construction\n\nOutput dictionary\n-----------------\n`pred`: Tensor[B, horizon, C, H, W]\n`latent`: final latent state\n`latent_seq`: Tensor[B, horizon, D, H, W]\n`diagnostics`: scalar tensors for energy drift, SOC strength, BER strength, etc.\n\nAuthoring note\n--------------\nThis is a cleaned reference architecture. Exact Colab benchmark implementations\nmay contain additional data-generation, plotting, logging, and ablation utilities.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n@dataclass\nclass CIDMConfig:\n    channels: int = 1\n    hist_len: int = 4\n    horizon: int = 6\n    latent_dim: int = 32\n    hidden: int = 64\n    port_channels: int = 1\n    use_phase_encoder: bool = True\n    use_time_encoder: bool = True\n    use_hidden_reservoir: bool = True\n    use_flux: bool = True\n    use_reverse_diagnostics: bool = True\n    # CIDM-BER options\n    soc_modes1: int = 8\n    soc_modes2: int = 8\n    ber_add_scale: float = 0.05\n    ber_enabled: bool = True\n    soc_enabled: bool = True\n\n\ndef count_parameters(model: nn.Module) -> int:\n    """Return the number of trainable parameters."""\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef conv_block(in_ch: int, out_ch: int, hidden: int, groups: int = 1) -> nn.Sequential:\n    return nn.Sequential(\n        nn.Conv2d(in_ch, hidden, 3, padding=1),\n        nn.GroupNorm(groups, hidden),\n        nn.SiLU(),\n        nn.Conv2d(hidden, hidden, 3, padding=1),\n        nn.GroupNorm(groups, hidden),\n        nn.SiLU(),\n        nn.Conv2d(hidden, out_ch, 1),\n    )\n\n\nclass SpaceEncoder(nn.Module):\n    """Encode the latest observed spatial field into latent information state."""\n\n    def __init__(self, channels: int, latent_dim: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(channels, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass TimeHistoryEncoder(nn.Module):\n    """Encode a finite observation history as temporal context."""\n\n    def __init__(self, channels: int, hist_len: int, latent_dim: int):\n        super().__init__()\n        self.hist_len = hist_len\n        self.net = nn.Sequential(\n            nn.Conv2d(channels * hist_len, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, history: torch.Tensor) -> torch.Tensor:\n        # history: B, L, C, H, W\n        b, l, c, h, w = history.shape\n        if l != self.hist_len:\n            raise ValueError(f"Expected hist_len={self.hist_len}, got {l}.")\n        return self.net(history.reshape(b, l * c, h, w))\n\n\nclass PhaseSpectralEncoder(nn.Module):\n    """FFT-based phase/amplitude encoder projected back to the spatial latent grid."""\n\n    def __init__(self, channels: int, latent_dim: int):\n        super().__init__()\n        self.freq_net = nn.Sequential(\n            nn.Conv2d(channels * 3, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n        )\n        self.spatial_net = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        z = torch.fft.fft2(x, dim=(-2, -1), norm="ortho")\n        amp = torch.log1p(torch.abs(z))\n        feats = torch.cat([z.real, z.imag, amp], dim=1)\n        f = self.freq_net(feats)\n        # Return to the spatial support while retaining frequency-aware features.\n        f_complex = torch.complex(f, torch.zeros_like(f))\n        spatial = torch.fft.ifft2(f_complex, dim=(-2, -1), norm="ortho").real\n        return self.spatial_net(spatial)\n\n\nclass FieldDecoder(nn.Module):\n    """Decode latent information state to physical/observed field."""\n\n    def __init__(self, latent_dim: int, channels: int, hidden: int = 64):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(latent_dim, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, channels, 1),\n        )\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.net(psi)\n\n\nclass SpectralFieldDecoder(nn.Module):\n    """Small spectral residual decoder used by CIDM for phase-aware readout."""\n\n    def __init__(self, latent_dim: int, channels: int):\n        super().__init__()\n        self.mix = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, channels, 1),\n        )\n        self.scale = nn.Parameter(torch.tensor(-2.2))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        z = torch.fft.fft2(psi, dim=(-2, -1), norm="ortho")\n        # Simple low-pass emphasis for phase-stable readout.\n        H, W = psi.shape[-2:]\n        ky = torch.fft.fftfreq(H, device=psi.device).view(1, 1, H, 1)\n        kx = torch.fft.fftfreq(W, device=psi.device).view(1, 1, 1, W)\n        filt = torch.exp(-6.0 * (kx ** 2 + ky ** 2))\n        low = torch.fft.ifft2(z * filt, dim=(-2, -1), norm="ortho").real\n        return torch.sigmoid(self.scale) * self.mix(low)\n\n\nclass AntisymmetricFlux(nn.Module):\n    """Conservative local exchange field.\n\n    For each neighbor pair, the edge update is constructed as f(a,b)-f(b,a),\n    then added to one side and subtracted from the other. This encodes the idea\n    that local exchanges can redistribute information without creating it.\n    """\n\n    def __init__(self, latent_dim: int, hidden: int):\n        super().__init__()\n        self.edge_net = nn.Sequential(\n            nn.Conv2d(2 * latent_dim, hidden, 1),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 1),\n        )\n\n    def edge(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:\n        return self.edge_net(torch.cat([a, b], dim=1)) - self.edge_net(torch.cat([b, a], dim=1))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        out = torch.zeros_like(psi)\n        # x direction\n        right = torch.roll(psi, shifts=-1, dims=-1)\n        fx = self.edge(psi, right)\n        out = out + fx - torch.roll(fx, shifts=1, dims=-1)\n        # y direction; safe for H=1 because roll is identity and antisymmetry cancels.\n        down = torch.roll(psi, shifts=-1, dims=-2)\n        fy = self.edge(psi, down)\n        out = out + fy - torch.roll(fy, shifts=1, dims=-2)\n        return out\n\n\nclass SkewChannelRotation(nn.Module):\n    """Learned skew-symmetric channel mixing for approximately conservative flow."""\n\n    def __init__(self, latent_dim: int):\n        super().__init__()\n        self.raw = nn.Parameter(torch.randn(latent_dim, latent_dim) * 0.02)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        A = self.raw - self.raw.t()\n        return torch.einsum("ij,bjhw->bihw", A, psi)\n\n\nclass HiddenReservoirTransfer(nn.Module):\n    """Internal hidden reservoir transfer for unobserved latent exchange."""\n\n    def __init__(self, latent_dim: int, hidden: int):\n        super().__init__()\n        self.net = conv_block(latent_dim, latent_dim, hidden)\n        self.scale = nn.Parameter(torch.tensor(-2.0))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return torch.sigmoid(self.scale) * self.net(psi)\n\n\nclass ClosedInformationDynamicsCore(nn.Module):\n    """CIDM closed latent vector field."""\n\n    def __init__(self, latent_dim: int, hidden: int, use_flux: bool = True, use_hidden_reservoir: bool = True):\n        super().__init__()\n        self.use_flux = use_flux\n        self.use_hidden_reservoir = use_hidden_reservoir\n        self.skew = SkewChannelRotation(latent_dim)\n        self.flux = AntisymmetricFlux(latent_dim, hidden)\n        self.hidden = HiddenReservoirTransfer(latent_dim, hidden)\n        self.local = conv_block(latent_dim, latent_dim, hidden)\n        self.step_scale = nn.Parameter(torch.tensor(0.18))\n\n    def vector_field(self, psi: torch.Tensor) -> torch.Tensor:\n        dpsi = self.skew(psi)\n        if self.use_flux:\n            dpsi = dpsi + self.flux(psi)\n        if self.use_hidden_reservoir:\n            dpsi = dpsi + 0.25 * self.hidden(psi)\n        # A small local residual stabilizes non-Hamiltonian observed fields.\n        dpsi = dpsi + 0.10 * self.local(psi)\n        return dpsi\n\n    def forward(self, psi: torch.Tensor, dt_scale: float = 1.0) -> torch.Tensor:\n        alpha = torch.tanh(self.step_scale) * dt_scale\n        return psi + alpha * self.vector_field(psi)\n\n\nclass LatentSpectralOperatorCore(nn.Module):\n    """Latent-domain spectral operator for CIDM-BER.\n\n    This is a residual spectral vector field inside the latent dynamics, not an\n    external FNO baseline. It learns complex low-frequency mode mixing and maps\n    it back to the latent grid as a global correction to local closed dynamics.\n    """\n\n    def __init__(self, latent_dim: int, hidden: int = 64, modes1: int = 8, modes2: int = 8):\n        super().__init__()\n        self.latent_dim = latent_dim\n        self.modes1 = modes1\n        self.modes2 = modes2\n        scale = 1.0 / max(1, latent_dim)\n        self.weight_pos = nn.Parameter(scale * torch.randn(latent_dim, latent_dim, modes1, modes2, 2))\n        self.weight_neg = nn.Parameter(scale * torch.randn(latent_dim, latent_dim, modes1, modes2, 2))\n        self.pre = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n        )\n        self.post = nn.Sequential(\n            nn.Conv2d(latent_dim, hidden, 1),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 1),\n        )\n        self.logit_scale = nn.Parameter(torch.tensor(-1.8))\n\n    @staticmethod\n    def compl_mul2d(x: torch.Tensor, wri: torch.Tensor) -> torch.Tensor:\n        w = torch.view_as_complex(wri.contiguous())\n        return torch.einsum("bihw,iohw->bohw", x, w)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        B, D, H, W = psi.shape\n        x = self.pre(psi)\n        x_ft = torch.fft.rfft2(x, dim=(-2, -1), norm="ortho")\n        out_ft = torch.zeros(B, D, H, W // 2 + 1, dtype=torch.cfloat, device=psi.device)\n        m1 = min(self.modes1, H)\n        m2 = min(self.modes2, W // 2 + 1)\n        out_ft[:, :, :m1, :m2] = self.compl_mul2d(x_ft[:, :, :m1, :m2], self.weight_pos[:, :, :m1, :m2])\n        if H > 1:\n            out_ft[:, :, -m1:, :m2] = self.compl_mul2d(x_ft[:, :, -m1:, :m2], self.weight_neg[:, :, :m1, :m2])\n        y = torch.fft.irfft2(out_ft, s=(H, W), dim=(-2, -1), norm="ortho")\n        return torch.sigmoid(self.logit_scale) * self.post(y)\n\n\nclass BoundaryEnergyReservoir(nn.Module):\n    """Boundary Energy Reservoir (BER) residual closure layer.\n\n    BER is a virtual external energy reservoir, not a full external-state model.\n    It learns a minimal residual exchange field that accounts for open-system\n    effects not captured by the internal closed dynamics.\n    """\n\n    def __init__(self, latent_dim: int, port_channels: int, hidden: int):\n        super().__init__()\n        self.port_channels = max(1, port_channels)\n        in_ch = latent_dim * 2 + self.port_channels\n        self.net = nn.Sequential(\n            nn.Conv2d(in_ch, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 3, padding=1),\n        )\n        mask_hidden = max(8, hidden // 2)\n        self.mask = nn.Sequential(\n            nn.Conv2d(in_ch, mask_hidden, 3, padding=1),\n            nn.SiLU(),\n            nn.Conv2d(mask_hidden, 1, 1),\n            nn.Sigmoid(),\n        )\n        self.logit_scale = nn.Parameter(torch.tensor(-3.0))\n\n    def _port_to_grid(self, psi: torch.Tensor, port: Optional[torch.Tensor]) -> torch.Tensor:\n        B, _, H, W = psi.shape\n        if port is None or port.numel() == 0:\n            return torch.zeros(B, self.port_channels, H, W, device=psi.device, dtype=psi.dtype)\n        if port.dim() != 4:\n            raise ValueError("port must be Tensor[B, C_port, H, W] for one step.")\n        p = port\n        if p.shape[-2:] != (H, W):\n            p = F.interpolate(p, size=(H, W), mode="bilinear", align_corners=False)\n        if p.shape[1] < self.port_channels:\n            pad = torch.zeros(B, self.port_channels - p.shape[1], H, W, device=psi.device, dtype=psi.dtype)\n            p = torch.cat([p, pad], dim=1)\n        elif p.shape[1] > self.port_channels:\n            p = p[:, : self.port_channels]\n        return p\n\n    def forward(\n        self,\n        psi: torch.Tensor,\n        internal_delta: torch.Tensor,\n        port: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        p = self._port_to_grid(psi, port)\n        x = torch.cat([psi, internal_delta, p], dim=1)\n        raw = torch.tanh(self.net(x))\n        mask = self.mask(x)\n        scale = torch.sigmoid(self.logit_scale)\n        ber_delta = scale * mask * raw\n        info = {\n            "ber_strength": ber_delta.abs().mean(),\n            "ber_mask_mean": mask.mean(),\n            "ber_scale": scale,\n        }\n        return ber_delta, info\n\n\nclass CIDM(nn.Module):\n    """Closed Information Dynamics Model.\n\n    General-purpose主体架构. This version corresponds to the stabilized closed\n    dynamics backbone: spatial information encoding, temporal history encoding,\n    phase-spectral feature encoding, conservative local exchange, internal hidden\n    reservoir transfer, and autoregressive latent rollout.\n    """\n\n    def __init__(self, config: CIDMConfig):\n        super().__init__()\n        self.config = config\n        D = config.latent_dim\n        H = config.hidden\n        self.space_encoder = SpaceEncoder(config.channels, D)\n        self.time_encoder = TimeHistoryEncoder(config.channels, config.hist_len, D)\n        self.phase_encoder = PhaseSpectralEncoder(config.channels, D)\n        fuse_inputs = D + (D if config.use_time_encoder else 0) + (D if config.use_phase_encoder else 0)\n        self.fuse = nn.Sequential(nn.Conv2d(fuse_inputs, D, 1), nn.SiLU(), nn.Conv2d(D, D, 1))\n        self.core = ClosedInformationDynamicsCore(D, H, use_flux=config.use_flux, use_hidden_reservoir=config.use_hidden_reservoir)\n        self.decoder = FieldDecoder(D, config.channels, H)\n        self.spectral_decoder = SpectralFieldDecoder(D, config.channels)\n\n    def encode(self, history: torch.Tensor) -> torch.Tensor:\n        last = history[:, -1]\n        feats = [self.space_encoder(last)]\n        if self.config.use_time_encoder:\n            feats.append(self.time_encoder(history))\n        if self.config.use_phase_encoder:\n            feats.append(self.phase_encoder(last))\n        return self.fuse(torch.cat(feats, dim=1))\n\n    def decode(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.decoder(psi) + self.spectral_decoder(psi)\n\n    def step(self, psi: torch.Tensor, step_index: int = 0, port: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        psi_next = self.core(psi)\n        return psi_next, {"soc_strength": torch.zeros((), device=psi.device), "ber_strength": torch.zeros((), device=psi.device)}\n\n    def _select_port_step(self, port: Optional[torch.Tensor], t: int) -> Optional[torch.Tensor]:\n        if port is None:\n            return None\n        if port.dim() == 5:\n            return port[:, min(t, port.shape[1] - 1)]\n        return port\n\n    def forward(\n        self,\n        history: torch.Tensor,\n        horizon: Optional[int] = None,\n        port: Optional[torch.Tensor] = None,\n        target: Optional[torch.Tensor] = None,\n    ) -> Dict[str, torch.Tensor]:\n        horizon = int(horizon or self.config.horizon)\n        psi0 = self.encode(history)\n        psi = psi0\n        preds = []\n        latent_seq = []\n        soc_strengths, ber_strengths = [], []\n        energy0 = (psi0 ** 2).mean(dim=(1, 2, 3))\n        energies = []\n        for t in range(horizon):\n            psi, info = self.step(psi, t, self._select_port_step(port, t))\n            y = self.decode(psi)\n            preds.append(y)\n            latent_seq.append(psi)\n            energies.append((psi ** 2).mean(dim=(1, 2, 3)))\n            soc_strengths.append(info.get("soc_strength", torch.zeros((), device=psi.device)))\n            ber_strengths.append(info.get("ber_strength", torch.zeros((), device=psi.device)))\n        pred = torch.stack(preds, dim=1)\n        latent = torch.stack(latent_seq, dim=1)\n        energy = torch.stack(energies, dim=1)\n        diagnostics = {\n            "energy_drift": (energy[:, -1] - energy0).abs().mean(),\n            "soc_strength": torch.stack([s if s.dim() else s.expand(1) for s in soc_strengths]).mean(),\n            "ber_strength": torch.stack([b if b.dim() else b.expand(1) for b in ber_strengths]).mean(),\n        }\n        if target is not None:\n            diagnostics["rmse"] = torch.sqrt(F.mse_loss(pred, target))\n            diagnostics["mae"] = F.l1_loss(pred, target)\n        return {"pred": pred, "latent": psi, "latent_seq": latent, "diagnostics": diagnostics}\n\n\nclass CIDM_BER(CIDM):\n    """CIDM with Spectral Operator Core and Boundary Energy Reservoir.\n\n    Physics-enhanced variant for systems with salient wave/flow propagation,\n    quasi-closed dynamics, boundary exchange, or energy closure requirements.\n    """\n\n    def __init__(self, config: CIDMConfig):\n        super().__init__(config)\n        self.soc = LatentSpectralOperatorCore(\n            config.latent_dim,\n            hidden=config.hidden,\n            modes1=config.soc_modes1,\n            modes2=config.soc_modes2,\n        )\n        self.ber = BoundaryEnergyReservoir(config.latent_dim, config.port_channels, config.hidden)\n        self.ber_add_scale = float(config.ber_add_scale)\n        self.soc_enabled = bool(config.soc_enabled)\n        self.ber_enabled = bool(config.ber_enabled)\n\n    def step(self, psi: torch.Tensor, step_index: int = 0, port: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        psi_int = self.core(psi)\n        soc_delta = self.soc(psi) if self.soc_enabled else torch.zeros_like(psi)\n        psi_int = psi_int + soc_delta\n        internal_delta = psi_int - psi\n        if self.ber_enabled:\n            ber_delta, ber_info = self.ber(psi, internal_delta.detach(), port)\n            psi_next = psi_int + self.ber_add_scale * ber_delta\n        else:\n            ber_delta = torch.zeros_like(psi)\n            ber_info = {"ber_strength": torch.zeros((), device=psi.device), "ber_mask_mean": torch.zeros((), device=psi.device)}\n            psi_next = psi_int\n        info = {\n            "soc_strength": soc_delta.abs().mean(),\n            "ber_strength": ber_info["ber_strength"],\n            "ber_mask_mean": ber_info.get("ber_mask_mean", torch.zeros((), device=psi.device)),\n            "ber_budget_proxy": (ber_delta.mean(dim=(1, 2, 3)) - internal_delta.mean(dim=(1, 2, 3))).abs().mean(),\n        }\n        return psi_next, info\n\n\ndef build_cidm(config: Optional[CIDMConfig] = None) -> CIDM:\n    return CIDM(config or CIDMConfig())\n\n\ndef build_cidm_ber(config: Optional[CIDMConfig] = None) -> CIDM_BER:\n    cfg = config or CIDMConfig()\n    return CIDM_BER(cfg)\n\n\n# Backward-compatible aliases for previous internal experiment names.\nClosedInformationDynamicsModel = CIDM\nClosedInformationDynamicsBERModel = CIDM_BER\nSTICM_V3_FULL_ALIAS = CIDM\nSTICM_SOC_BER_ALIAS = CIDM_BER\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n    cfg = CIDMConfig(channels=1, hist_len=4, horizon=1, latent_dim=4, hidden=8, port_channels=1)\n    x = torch.randn(1, cfg.hist_len, cfg.channels, 4, 4)\n    port = torch.randn(1, cfg.port_channels, 4, 4)\n    for cls in [CIDM, CIDM_BER]:\n        model = cls(cfg)\n        out = model(x, port=port)\n        print(cls.__name__, out["pred"].shape, "params=", count_parameters(model), out["diagnostics"])\n'

EXPECTED_CORE_SHA256 = "e5e26b35765301e9ecd25aec5a6c507395ff370dca11c820f540ff0f1e79f510"
EXPECTED_MANIFEST_SHA256 = "9508dd000005a154d3d3fa4437602d4214a6266d630bf13af855e5517649ecbd"
EXPECTED_COMPLETE_ARCHIVE_SHA256 = "f9b1cf4ff7630f5540bb3adde20e4c4b81fff3218d9325ea3105bf28b1d97665"

core_file=FROZEN/"cidm_foundation_core_v1.py"
manifest_file=FROZEN/"CIDM_Foundation_Core_v1_0_FREEZE_MANIFEST.json"
complete_file=FROZEN/"cidm_complete_archived_topology.py"

core_file.write_text(CIDM_CORE_SOURCE,encoding="utf-8")
manifest_file.write_text(CIDM_FREEZE_MANIFEST,encoding="utf-8")
complete_file.write_text(CIDM_COMPLETE_ARCHIVE_SOURCE,encoding="utf-8")

core_sha=sha256_file(core_file)
manifest_sha=sha256_file(manifest_file)
complete_sha=sha256_file(complete_file)

print("CIDM Core SHA256:",core_sha)
print("Core Manifest SHA256:",manifest_sha)
print("CIDM-Complete archive SHA256:",complete_sha)

assert core_sha==EXPECTED_CORE_SHA256
assert manifest_sha==EXPECTED_MANIFEST_SHA256
assert complete_sha==EXPECTED_COMPLETE_ARCHIVE_SHA256

import importlib.util

spec=importlib.util.spec_from_file_location("cidm_foundation_core_v1",core_file)
cidm_mod=importlib.util.module_from_spec(spec)
sys.modules["cidm_foundation_core_v1"]=cidm_mod
spec.loader.exec_module(cidm_mod)

CIDMFoundationCore=cidm_mod.CIDMFoundationCore
CIDMFoundationConfig=cidm_mod.CIDMFoundationConfig
build_cidm_foundation_core=cidm_mod.build_cidm_foundation_core

spec2=importlib.util.spec_from_file_location("cidm_complete_archive",complete_file)
cidm_complete_mod=importlib.util.module_from_spec(spec2)
sys.modules["cidm_complete_archive"]=cidm_complete_mod
spec2.loader.exec_module(cidm_complete_mod)

CIDMCompleteConfig=cidm_complete_mod.CIDMConfig
CIDMCompleteArchived=cidm_complete_mod.CIDM_BER

# Core smoke
m=build_cidm_foundation_core(
    spatial_dim=2,channels=1,hist_len=4,horizon=6,
    latent_dim=16,hidden=32
).to(DEVICE)
x=torch.randn(2,4,1,24,24,device=DEVICE)
with torch.no_grad():
    y=m(x)["pred"]
assert y.shape==(2,6,1,24,24)
del m,x,y

# Complete 2D smoke
cfg=CIDMCompleteConfig(
    channels=1,hist_len=4,horizon=6,
    latent_dim=8,hidden=16,port_channels=1,
    use_phase_encoder=True,use_time_encoder=True,
    use_hidden_reservoir=True,use_flux=True,
    soc_modes1=2,soc_modes2=4,
    ber_add_scale=0.05,ber_enabled=True,soc_enabled=True,
)
m=CIDMCompleteArchived(cfg).to(DEVICE)
x=torch.randn(2,4,1,24,24,device=DEVICE)
with torch.no_grad():
    y=m(x,horizon=6,port=None)["pred"]
assert y.shape==(2,6,1,24,24)
del m,x,y

# Complete 1D-via-H=1 smoke
cfg=CIDMCompleteConfig(
    channels=1,hist_len=4,horizon=6,
    latent_dim=8,hidden=16,port_channels=1,
    use_phase_encoder=True,use_time_encoder=True,
    use_hidden_reservoir=True,use_flux=True,
    soc_modes1=1,soc_modes2=4,
    ber_add_scale=0.05,ber_enabled=True,soc_enabled=True,
)
m=CIDMCompleteArchived(cfg).to(DEVICE)
x=torch.randn(2,4,1,1,64,device=DEVICE)
with torch.no_grad():
    y=m(x,horizon=6,port=None)["pred"]
assert y.shape==(2,6,1,1,64)
del m,x,y
torch.cuda.empty_cache()

COMPLETE_CONTRACT={
    "benchmark_name":"CIDM-Complete",
    "executable_mapping":"cidm_architectures.py::CIDM_BER",
    "archive_sha256":complete_sha,
    "components":[
        "SpaceEncoder","TimeHistoryEncoder","PhaseSpectralEncoder",
        "SkewChannelRotation","AntisymmetricFlux","HiddenReservoirTransfer",
        "LocalResidual","LatentSpectralOperatorCore","BoundaryEnergyReservoir",
        "FieldDecoder","SpectralFieldDecoder","PureLatentRollout"
    ],
    "known_forcing_port":"disabled / None because PDEBench headline tasks provide no common forcing field",
    "static_fields":"not supplied; absent from archived executable implementation",
    "1d_protocol":"lift to H=1, run exact archived Conv2D implementation, squeeze H=1",
    "future_target_at_inference":"forbidden",
    "note":"Figure-level State-Observation Closure is not invented here; archived executable SOC is LatentSpectralOperatorCore."
}
save_json(COMPLETE_CONTRACT,OUT/"CIDM_Complete_Benchmark_Contract.json")

REPORT["cidm_freeze"]={
    "core_sha256":core_sha,
    "manifest_sha256":manifest_sha,
    "exact":True,
}
REPORT["cidm_complete_freeze"]={
    "archive_sha256":complete_sha,
    "exact":True,
    "contract":COMPLETE_CONTRACT,
}

print("FROZEN CIDM CORE: PASS")
print("FROZEN CIDM-COMPLETE ARCHIVE: PASS")



# 2. Exact official source checkout

固定此前 provenance qualification 已验证过的 exact commit：

```text
PDEBench       4ff3e3a4aa1561721b5571fa3a048a0a463e0568
NeuralOperator 00b7d86f8d74ff0af55da53eb585fe26df9c71f0
Transolver     75e0f67643806a81cd1d3f6adc88dd8c02416fe7
```

正式 Benchmark 不追随 `main/master`，防止实验期间 upstream 漂移。


In [ ]:

# ============================================================
# 2. Exact source checkout + official imports
# ============================================================
LOCKED={
    "pdebench":{
        "url":"https://github.com/pdebench/PDEBench.git",
        "sha":"4ff3e3a4aa1561721b5571fa3a048a0a463e0568",
    },
    "neuraloperator":{
        "url":"https://github.com/neuraloperator/neuraloperator.git",
        "sha":"00b7d86f8d74ff0af55da53eb585fe26df9c71f0",
    },
    "transolver":{
        "url":"https://github.com/thuml/Transolver.git",
        "sha":"75e0f67643806a81cd1d3f6adc88dd8c02416fe7",
    },
}
checkout=[]
for rid,m in LOCKED.items():
    dst=SRC/rid
    if dst.exists():
        rc,head=run(["git","rev-parse","HEAD"],cwd=dst,timeout=60)
        if rc==0 and head.strip()==m["sha"]:
            print("[reuse source]",rid,head.strip())
            checkout.append({"id":rid,"head":head.strip(),"exact":True})
            continue
        shutil.rmtree(dst)

    rc,o=run(["git","clone","--filter=blob:none","--no-checkout",m["url"],str(dst)])
    if rc: raise RuntimeError(f"{rid} clone failed\n{o[-3000:]}")
    rc,o=run(["git","fetch","origin",m["sha"]],cwd=dst)
    if rc: raise RuntimeError(f"{rid} fetch failed\n{o[-3000:]}")
    rc,o=run(["git","checkout","--detach",m["sha"]],cwd=dst)
    if rc: raise RuntimeError(f"{rid} checkout failed\n{o[-3000:]}")
    _,head=run(["git","rev-parse","HEAD"],cwd=dst)
    head=head.strip()
    checkout.append({"id":rid,"head":head,"exact":head==m["sha"]})
    print(rid,head,"PASS" if head==m["sha"] else "FAIL")

assert all(x["exact"] for x in checkout)
pd.DataFrame(checkout).to_csv(OUT/"source_checkout.csv",index=False)

# Dependencies needed by the exact official cores.
rc,o=run([
    sys.executable,"-m","pip","install","-q",
    "tensorly","tensorly-torch","einops","timm","scipy"
],timeout=1200)
if rc:
    raise RuntimeError(o[-5000:])

sys.path.insert(0,str(SRC/"pdebench"))
sys.path.insert(0,str(SRC/"neuraloperator"))
TRANS_BENCH=SRC/"transolver"/"PDE-Solving-StandardBenchmark"
sys.path.insert(0,str(TRANS_BENCH))

from pdebench.models.fno.fno import FNO1d as PDE_FNO1d, FNO2d as PDE_FNO2d
from pdebench.models.unet.unet import UNet1d as PDE_UNet1d, UNet2d as PDE_UNet2d
from neuralop.models import FNO as NeuralOpFNO
from model.Transolver_Irregular_Mesh import Model as TransolverIrregular
from model.Transolver_Structured_Mesh_2D import Model as Transolver2D

print("OFFICIAL IMPORTS: PASS")
REPORT["source_checkout"]=checkout



# 3. Public PDEBench data and deterministic frozen windows

本轮使用比 Pilot 更大的固定 public subset：

默认：
- train trajectories = 256
- validation = 64
- test = 64
- 每 trajectory 2 个 deterministic windows

若数据集不足，则自动使用可用最大数量。

仍保留：
- 前 10% trajectory 为 test pool；
- 剩余尾部 10% 为 validation pool；
- 其余为 train pool；
- 每个 pool 内做 deterministic evenly-spaced selection。

所有模型读取**同一个 frozen cache**。



### 3.1 Hugging Face Dataset repository resume — FIX v3

上一版没有成功恢复进度的原因已经定位：

```text
[HF] enabled: CIDM-v1
[HF RESTORE] scanning dataset repo: CIDM-v1
...
https://huggingface.co/api/datasets/CIDM-v1/tree/main
404 Repository Not Found
```

也就是说，`repo_type="dataset"` **其实已经是正确的**，真正的错误是：

```text
HF_REPO_ID = CIDM-v1
```

这个旧 Secret / 环境变量覆盖了完整仓库名：

```text
wuff-mann/CIDM-v1
```

`create_repo()` 可以根据登录用户补全短名称，但后续 `list_repo_files()` 使用短名称时不会自动补成
`wuff-mann/CIDM-v1`，于是访问了不存在的：

```text
datasets/CIDM-v1
```

---

## v3 修复原则

本实验的跨 Colab 状态仓库现在**强制固定为 Dataset repo**：

```text
wuff-mann/CIDM-v1
```

不再读取旧的 `HF_REPO_ID=CIDM-v1` 作为恢复目标。

所有 Hub API 调用均显式：

```python
repo_type="dataset"
```

并在真正恢复前先执行：

```python
api.dataset_info("wuff-mann/CIDM-v1")
```

验证它确实是 Dataset 仓库。

---

## 旧上传目录自动识别

你之前执行的是：

```python
upload_folder(folder_path="/content", repo_type="dataset")
```

因此远端可能存在：

```text
CIDM_Foundation_A1_R2_R1_PUBLICATION_RESULTS/...
CIDM_PDEBench_Data/...
CIDM_PDEBench_A1R2_Cache/...
```

v3 不再依赖固定目录层级。

只要远端任意路径的文件名满足：

```text
publication__*_result.json
publication__*_history.csv
publication__*_latest.pt
```

就自动映射回本地正确位置：

```text
OUT/
OUT/checkpoints/
```

所以即使历史上传后来又多套了一层 `/content`、结果目录或其他前缀，也可以识别。

---

## 恢复硬 Gate

如果 HF Dataset 中检测到旧实验 result/checkpoint，
但恢复完成后本地仍然是：

```text
completed=0
checkpoints=0
```

脚本会**直接停止**，绝不悄悄从头训练浪费 Colab 时间。

只有恢复审计通过后，才进入 180-run 主循环。


In [ ]:
# ============================================================
# 3.1 HF DATASET resume FIX v3
#     Canonical repo: wuff-mann/CIDM-v1
# ============================================================
import os, sys, json, time, shutil, subprocess, re
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------------
# Canonical Dataset repository.
# IMPORTANT: do not let the stale short HF_REPO_ID="CIDM-v1"
# override this fully-qualified Dataset repo.
# ------------------------------------------------------------------
HF_DATASET_REPO_ID = os.getenv(
    "CIDM_HF_DATASET_REPO",
    "wuff-mann/CIDM-v1",
).strip()
HF_REPO_TYPE = "dataset"
HF_PROGRESS_PREFIX = "A1_R2_R1_Publication_500K"

HF_RESTORE_DATA = os.getenv("CIDM_HF_RESTORE_DATA", "1") == "1"
HF_CHECKPOINT_EVERY = int(os.getenv("CIDM_HF_CHECKPOINT_EVERY", "500"))
HF_MAX_RETRIES = int(os.getenv("CIDM_HF_MAX_RETRIES", "4"))

# Existing progress is expected in this project. Set to 0 only if intentionally
# starting a brand-new repository.
HF_EXPECT_EXISTING_PROGRESS = (
    os.getenv("CIDM_HF_EXPECT_EXISTING_PROGRESS", "1") == "1"
)

# Local-only fallback is OFF by default because silently restarting 180 runs
# would waste compute. Set to 1 only for deliberate offline testing.
HF_ALLOW_LOCAL_ONLY = os.getenv("CIDM_HF_ALLOW_LOCAL_ONLY", "0") == "1"

# Token: Colab Secret first, environment second.
try:
    from google.colab import userdata
    HF_TOKEN_RESOLVED = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN_RESOLVED = os.getenv("HF_TOKEN")

# Show and ignore the legacy ambiguous setting that caused v2 to fail.
try:
    _legacy_repo_secret = (
        userdata.get("HF_REPO_ID")
        if "userdata" in globals()
        else os.getenv("HF_REPO_ID")
    )
except Exception:
    _legacy_repo_secret = os.getenv("HF_REPO_ID")

if _legacy_repo_secret and _legacy_repo_secret != HF_DATASET_REPO_ID:
    print(
        f"[HF CONFIG] legacy HF_REPO_ID={_legacy_repo_secret!r} is ignored "
        f"for resume. Canonical Dataset repo = {HF_DATASET_REPO_ID!r}"
    )

# Keep downstream code consistent, but with the FULL canonical ID.
HF_REPO_ID_RESOLVED = HF_DATASET_REPO_ID
os.environ["HF_REPO_ID"] = HF_REPO_ID_RESOLVED
os.environ["CIDM_HF_DATASET_REPO"] = HF_REPO_ID_RESOLVED

HF_ENABLED = False
HF_API = None
HF_REMOTE_FILES = []
HF_RESTORE_AUDIT = {}

_RUN_RESULT_RE = re.compile(r"^publication__.+__.+__s\d+_result\.json$")
_RUN_HISTORY_RE = re.compile(r"^publication__.+__.+__s\d+_history\.csv$")
_RUN_CKPT_RE = re.compile(r"^publication__.+__.+__s\d+_latest\.pt$")


def _utc_now():
    return datetime.now(timezone.utc).isoformat()


def _retry_call(fn, *args, **kwargs):
    last = None
    for attempt in range(1, HF_MAX_RETRIES + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last = e
            if attempt == HF_MAX_RETRIES:
                break
            wait = min(30, 2 ** attempt)
            print(
                f"[HF retry {attempt}/{HF_MAX_RETRIES}] "
                f"{type(e).__name__}: {e}; sleep {wait}s"
            )
            time.sleep(wait)
    raise last


def _normalize_remote_path(path: str) -> str:
    return str(path).replace("\\", "/").lstrip("/")


def _remote_to_local(remote_path: str):
    """
    Map ANY compatible historical repository layout back to the live local tree.

    Progress mapping is primarily filename-based, therefore it works for:
      - A1_R2_R1_Publication_500K/...
      - CIDM_Foundation_A1_R2_R1_PUBLICATION_RESULTS/...
      - content/CIDM_Foundation_A1_R2_R1_PUBLICATION_RESULTS/...
      - repo-root upload
      - any accidental extra parent directory
    """
    rp = _normalize_remote_path(remote_path)
    name = Path(rp).name

    # Highest priority: current standardized namespace.
    new_prefix = HF_PROGRESS_PREFIX + "/"
    if rp.startswith(new_prefix):
        rel = rp[len(new_prefix):]
        return OUT / rel, 100, "v3_standard"

    # Run artifacts: map by basename, independent of parent directories.
    if _RUN_RESULT_RE.match(name):
        return OUT / name, 80, "result_by_name"

    if _RUN_HISTORY_RE.match(name):
        return OUT / name, 80, "history_by_name"

    if _RUN_CKPT_RE.match(name):
        return CKPT / name, 80, "checkpoint_by_name"

    # Prediction examples can appear under arbitrary historical parents.
    if "prediction_examples/" in rp and name.endswith(".pt"):
        return PRED / name, 70, "prediction_example"

    # Cumulative progress files.
    if name in {
        "progress_manifest.json",
        "publication_run_results.csv",
        "publication_run_results_partial.csv",
    }:
        return OUT / name, 60, "progress_summary"

    # Optional input data/cache. Search for the directory marker anywhere
    # in the remote path, not only at repo root.
    for marker in ["CIDM_PDEBench_Data/", "CIDM_PDEBench_A1R2_Cache/"]:
        pos = rp.find(marker)
        if pos >= 0 and HF_RESTORE_DATA:
            suffix = rp[pos:]
            return Path("/content") / suffix, 40, "input_data"

    return None, -1, None


def _is_progress_remote_file(path: str):
    dst, _, kind = _remote_to_local(path)
    if dst is None:
        return False
    return kind != "input_data"


def _is_data_remote_file(path: str):
    dst, _, kind = _remote_to_local(path)
    return dst is not None and kind == "input_data"


def _smart_copy(src: Path, dst: Path):
    """
    Merge remote state without rolling local progress backwards.
    """
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if not dst.exists():
        shutil.copy2(src, dst)
        return "copied"

    # Results: COMPLETED always wins.
    if dst.name.endswith("_result.json"):
        try:
            local = json.loads(dst.read_text(encoding="utf-8"))
            remote = json.loads(src.read_text(encoding="utf-8"))

            ldone = local.get("status") == "COMPLETED"
            rdone = remote.get("status") == "COMPLETED"

            if ldone and not rdone:
                return "kept_local_completed"
            if rdone and not ldone:
                shutil.copy2(src, dst)
                return "remote_completed"

            # Same completion state: prefer the one with larger actual_updates.
            lu = int(local.get("actual_updates", -1) or -1)
            ru = int(remote.get("actual_updates", -1) or -1)
            if ru > lu:
                shutil.copy2(src, dst)
                return f"remote_result_updates_{ru}"
            return f"kept_local_result_updates_{lu}"
        except Exception:
            return "kept_local_result_parse_failure"

    # Checkpoints: largest optimizer step wins.
    if dst.suffix == ".pt" and "_latest" in dst.name:
        try:
            local = torch.load(dst, map_location="cpu", weights_only=False)
            remote = torch.load(src, map_location="cpu", weights_only=False)
            ls = int(local.get("step", -1))
            rs = int(remote.get("step", -1))
            if rs > ls:
                shutil.copy2(src, dst)
                return f"remote_checkpoint_step_{rs}"
            return f"kept_local_checkpoint_step_{ls}"
        except Exception:
            return "kept_local_checkpoint_parse_failure"

    # History: largest recorded step wins.
    if dst.name.endswith("_history.csv"):
        try:
            local = pd.read_csv(dst)
            remote = pd.read_csv(src)
            ls = int(local["step"].max()) if len(local) and "step" in local else -1
            rs = int(remote["step"].max()) if len(remote) and "step" in remote else -1
            if rs > ls:
                shutil.copy2(src, dst)
                return f"remote_history_step_{rs}"
            return f"kept_local_history_step_{ls}"
        except Exception:
            return "kept_local_history_parse_failure"

    # For summaries/examples, keep an existing local artifact.
    return "kept_local"


def scan_local_progress():
    rows = []
    for p in sorted(OUT.glob("publication__*_result.json")):
        try:
            x = json.loads(p.read_text(encoding="utf-8"))
            x["_file"] = p.name
            rows.append(x)
        except Exception:
            pass

    completed = [x for x in rows if x.get("status") == "COMPLETED"]
    failed = [x for x in rows if x.get("status") == "FAILED"]
    checkpoints = sorted(CKPT.glob("publication__*_latest.pt"))

    ck_steps = {}
    for p in checkpoints:
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            ck_steps[p.name] = int(ck.get("step", -1))
        except Exception:
            ck_steps[p.name] = -1

    return {
        "total_expected": len(SEEDS) * 3 * len(MODELS),
        "result_files": len(rows),
        "completed": len(completed),
        "failed_or_partial_results": len(failed),
        "latest_checkpoints": len(checkpoints),
        "checkpoint_steps": ck_steps,
        "completed_run_ids": sorted(
            f"publication__{x.get('model')}__{x.get('task')}__s{x.get('seed')}"
            for x in completed
        ),
    }


def write_progress_manifest(current_run=None, current_step=None, event=None):
    state = scan_local_progress()
    state.update({
        "experiment": "CIDM Foundation A1-R2-R1 Publication Confirmation",
        "hf_repo_id": HF_REPO_ID_RESOLVED,
        "hf_repo_type": HF_REPO_TYPE,
        "hf_prefix": HF_PROGRESS_PREFIX,
        "updated_at_utc": _utc_now(),
        "current_run": current_run,
        "current_step": current_step,
        "event": event,
    })

    path = OUT / "progress_manifest.json"
    path.write_text(
        json.dumps(state, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    rows = []
    for p in sorted(OUT.glob("publication__*_result.json")):
        try:
            rows.append(json.loads(p.read_text(encoding="utf-8")))
        except Exception:
            pass
    if rows:
        pd.DataFrame(rows).to_csv(
            OUT / "publication_run_results_partial.csv",
            index=False,
        )
    return path, state


def _repo_path(local_path: Path):
    rel = Path(local_path).resolve().relative_to(OUT.resolve()).as_posix()
    return f"{HF_PROGRESS_PREFIX}/{rel}"


def _hf_commit_files(local_files, message):
    if not HF_ENABLED:
        return False

    from huggingface_hub import CommitOperationAdd

    ops = []
    seen = set()
    for p in local_files:
        p = Path(p)
        if not p.exists() or not p.is_file():
            continue
        rp = _repo_path(p)
        if rp in seen:
            continue
        seen.add(rp)
        ops.append(
            CommitOperationAdd(
                path_in_repo=rp,
                path_or_fileobj=str(p),
            )
        )

    if not ops:
        return True

    _retry_call(
        HF_API.create_commit,
        repo_id=HF_REPO_ID_RESOLVED,
        repo_type="dataset",
        operations=ops,
        commit_message=message,
    )
    return True


def hf_sync_checkpoint(latest_path: Path, run_id: str, step: int):
    if not HF_ENABLED:
        return

    manifest_path, _ = write_progress_manifest(
        current_run=run_id,
        current_step=step,
        event="MID_RUN_CHECKPOINT",
    )

    try:
        _hf_commit_files(
            [latest_path, manifest_path],
            f"[A1-R2-R1] checkpoint {run_id} step={step}",
        )
        print(
            f"[HF DATASET CHECKPOINT] {run_id} step={step} "
            f"-> {HF_REPO_ID_RESOLVED}"
        )
    except Exception as e:
        print(
            f"[HF DATASET CHECKPOINT WARNING] "
            f"{run_id} step={step}: {e}"
        )


def hf_sync_finished_run(
    run_id: str,
    result_path: Path,
    history_path: Path,
    latest_path: Path,
):
    if not HF_ENABLED:
        return

    try:
        row = json.loads(Path(result_path).read_text(encoding="utf-8"))
        status = row.get("status", "UNKNOWN")
    except Exception:
        status = "UNKNOWN"

    manifest_path, state = write_progress_manifest(
        current_run=run_id,
        current_step=None,
        event=f"RUN_{status}",
    )

    files = [
        result_path,
        history_path,
        manifest_path,
        OUT / "publication_run_results_partial.csv",
    ]

    try:
        parts = run_id.split("__")
        model = parts[1]
        task = parts[2]
        seed = int(parts[3][1:])
        if seed == SEEDS[0]:
            pred_file = PRED / f"{task}__{model}.pt"
            if pred_file.exists():
                files.append(pred_file)
    except Exception:
        pass

    if status != "COMPLETED" and Path(latest_path).exists():
        files.append(latest_path)

    try:
        _hf_commit_files(
            files,
            (
                f"[A1-R2-R1] {status} {run_id} | "
                f"completed={state['completed']}/{state['total_expected']}"
            ),
        )
        print(
            f"[HF DATASET RUN SYNC] {run_id}: {status}; "
            f"completed={state['completed']}/{state['total_expected']}"
        )

        # Completed run no longer needs remote latest checkpoint.
        if status == "COMPLETED":
            remote_ck = _repo_path(Path(latest_path))
            try:
                _retry_call(
                    HF_API.delete_file,
                    path_in_repo=remote_ck,
                    repo_id=HF_REPO_ID_RESOLVED,
                    repo_type="dataset",
                    commit_message=(
                        f"[A1-R2-R1] cleanup completed checkpoint {run_id}"
                    ),
                )
            except Exception:
                pass

    except Exception as e:
        print(f"[HF DATASET RUN SYNC WARNING] {run_id}: {e}")


def _download_remote_file(remote_file: str, restore_root: Path):
    from huggingface_hub import hf_hub_download
    return Path(
        _retry_call(
            hf_hub_download,
            repo_id=HF_REPO_ID_RESOLVED,
            filename=remote_file,
            repo_type="dataset",
            revision="main",
            token=HF_TOKEN_RESOLVED,
            local_dir=str(restore_root),
        )
    )


def restore_progress_from_hf():
    """
    Restore old /content upload + v2/v3 incremental layouts from the canonical
    Hugging Face DATASET repository.
    """
    global HF_REMOTE_FILES, HF_RESTORE_AUDIT

    print("=" * 92)
    print("[HF DATASET RESTORE]")
    print("repo_id   :", HF_REPO_ID_RESOLVED)
    print("repo_type :", HF_REPO_TYPE)
    print("=" * 92)

    HF_REMOTE_FILES = _retry_call(
        HF_API.list_repo_files,
        repo_id=HF_REPO_ID_RESOLVED,
        repo_type="dataset",
        revision="main",
    )

    progress_files = [f for f in HF_REMOTE_FILES if _is_progress_remote_file(f)]
    data_files = [f for f in HF_REMOTE_FILES if _is_data_remote_file(f)]

    result_candidates = [
        f for f in progress_files
        if _RUN_RESULT_RE.match(Path(_normalize_remote_path(f)).name)
    ]
    ckpt_candidates = [
        f for f in progress_files
        if _RUN_CKPT_RE.match(Path(_normalize_remote_path(f)).name)
    ]
    history_candidates = [
        f for f in progress_files
        if _RUN_HISTORY_RE.match(Path(_normalize_remote_path(f)).name)
    ]

    print(f"remote files total        : {len(HF_REMOTE_FILES)}")
    print(f"progress candidates       : {len(progress_files)}")
    print(f"  result JSON candidates  : {len(result_candidates)}")
    print(f"  history CSV candidates  : {len(history_candidates)}")
    print(f"  checkpoint candidates   : {len(ckpt_candidates)}")
    print(f"input data/cache files    : {len(data_files)}")

    if progress_files:
        print("\n[HF RESTORE] sample compatible remote paths:")
        for f in progress_files[:25]:
            dst, _, kind = _remote_to_local(f)
            print(f"  {kind:20s} {f} -> {dst}")

    restore_root = Path("/content/_cidm_hf_dataset_restore")
    restore_root.mkdir(parents=True, exist_ok=True)

    actions = []
    restored_files = 0

    # Download ALL progress candidates. Duplicates from old/new layouts are
    # merged by _smart_copy (completed result / max checkpoint step / max history).
    #
    # Sort low-priority first, high-priority later so standardized v3 artifacts
    # get the final opportunity to replace older equivalents.
    mapped_progress = []
    for rf in progress_files:
        dst, priority, kind = _remote_to_local(rf)
        mapped_progress.append((priority, rf, dst, kind))
    mapped_progress.sort(key=lambda x: x[0])

    for priority, rf, dst, kind in mapped_progress:
        try:
            local = _download_remote_file(rf, restore_root)
            action = _smart_copy(local, dst)
            restored_files += 1
            actions.append({
                "remote": rf,
                "local": str(dst),
                "kind": kind,
                "priority": priority,
                "action": action,
            })
            if kind in {"result_by_name", "checkpoint_by_name", "v3_standard"}:
                if (
                    "_result.json" in rf
                    or "_latest.pt" in rf
                ):
                    print(f"  [restored] {rf} -> {dst.name} ({action})")
        except Exception as e:
            actions.append({
                "remote": rf,
                "local": str(dst),
                "kind": kind,
                "priority": priority,
                "action": f"ERROR: {repr(e)}",
            })
            print(f"  [restore warning] {rf}: {e}")

    # Restore only missing large data/cache files.
    data_restored = 0
    if HF_RESTORE_DATA:
        # Deduplicate data destinations, prefer shallower/newer mapping
        # only by exact destination (contents should be identical).
        unique_data = {}
        for rf in data_files:
            dst, priority, kind = _remote_to_local(rf)
            unique_data.setdefault(str(dst), (rf, dst))

        pending = [
            (rf, dst)
            for rf, dst in unique_data.values()
            if not dst.exists()
        ]
        if pending:
            print(
                f"\n[HF DATASET RESTORE] restoring "
                f"{len(pending)} missing PDEBench data/cache files..."
            )

        for rf, dst in pending:
            try:
                local = _download_remote_file(rf, restore_root)
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(local, dst)
                data_restored += 1
                print(f"  [data restored] {rf}")
            except Exception as e:
                print(f"  [data restore warning] {rf}: {e}")

    state = scan_local_progress()

    HF_RESTORE_AUDIT = {
        "repo_id": HF_REPO_ID_RESOLVED,
        "repo_type": "dataset",
        "remote_file_count": len(HF_REMOTE_FILES),
        "remote_progress_candidates": len(progress_files),
        "remote_result_candidates": len(result_candidates),
        "remote_history_candidates": len(history_candidates),
        "remote_checkpoint_candidates": len(ckpt_candidates),
        "restored_progress_files_checked": restored_files,
        "restored_data_files": data_restored,
        "local_after_restore": state,
        "actions": actions,
        "timestamp_utc": _utc_now(),
    }

    audit_path = OUT / "HF_DATASET_RESTORE_AUDIT.json"
    audit_path.write_text(
        json.dumps(HF_RESTORE_AUDIT, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("\n" + "=" * 92)
    print("[HF DATASET RESTORE COMPLETE]")
    print(
        f"local completed runs : "
        f"{state['completed']}/{state['total_expected']}"
    )
    print(
        f"local result files   : {state['result_files']}"
    )
    print(
        f"local checkpoints    : {state['latest_checkpoints']}"
    )
    if state["checkpoint_steps"]:
        for k,v in sorted(state["checkpoint_steps"].items()):
            print(f"  checkpoint {k}: step={v}")
    print("restore audit        :", audit_path)
    print("=" * 92)

    # HARD SAFETY GATE:
    # remote repo contains training progress, but nothing was reconstructed.
    if (
        (len(result_candidates) > 0 or len(ckpt_candidates) > 0)
        and state["result_files"] == 0
        and state["latest_checkpoints"] == 0
    ):
        raise RuntimeError(
            "HF Dataset contains prior experiment result/checkpoint files, "
            "but zero resumable progress was reconstructed locally. "
            "Training is BLOCKED to prevent an accidental restart. "
            "Inspect HF_DATASET_RESTORE_AUDIT.json and the printed remote paths."
        )

    if HF_EXPECT_EXISTING_PROGRESS:
        if len(result_candidates) == 0 and len(ckpt_candidates) == 0:
            raise RuntimeError(
                "CIDM_HF_EXPECT_EXISTING_PROGRESS=1, but no prior "
                "publication__*_result.json or publication__*_latest.pt "
                f"was detected in Dataset repo {HF_REPO_ID_RESOLVED}. "
                "Training is BLOCKED instead of silently restarting. "
                "If you intentionally want a fresh experiment, set "
                "CIDM_HF_EXPECT_EXISTING_PROGRESS=0."
            )

    return state


# ------------------------------------------------------------------
# Local path mapping self-test BEFORE contacting HF.
# ------------------------------------------------------------------
_mapping_tests = {
    "CIDM_Foundation_A1_R2_R1_PUBLICATION_RESULTS/"
    "publication__CIDM-Complete__diff_react_2d__s41_result.json":
        OUT/"publication__CIDM-Complete__diff_react_2d__s41_result.json",

    "some/extra/parent/checkpoints/"
    "publication__PDEBench-FNO__burgers__s73_latest.pt":
        CKPT/"publication__PDEBench-FNO__burgers__s73_latest.pt",

    "content/CIDM_PDEBench_Data/1D_Burgers_Sols_Nu0.01.hdf5":
        Path("/content/CIDM_PDEBench_Data/1D_Burgers_Sols_Nu0.01.hdf5"),

    f"{HF_PROGRESS_PREFIX}/checkpoints/"
    "publication__CIDM-Complete__swe_2d__s101_latest.pt":
        CKPT/"publication__CIDM-Complete__swe_2d__s101_latest.pt",
}

for remote, expected in _mapping_tests.items():
    got, _, _ = _remote_to_local(remote)
    assert got == expected, (remote, got, expected)

print("[HF DATASET PATH MAPPING SELF-TEST] PASS")


# ------------------------------------------------------------------
# Initialize canonical Dataset repo and restore.
# ------------------------------------------------------------------
if not HF_TOKEN_RESOLVED:
    msg = (
        "HF_TOKEN was not found. This benchmark is configured for cross-Colab "
        "resume from Dataset repo wuff-mann/CIDM-v1."
    )
    if HF_ALLOW_LOCAL_ONLY:
        print("[HF LOCAL-ONLY WARNING]", msg)
    else:
        raise RuntimeError(msg)

else:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"],
            check=True,
        )

        from huggingface_hub import HfApi

        HF_API = HfApi(token=HF_TOKEN_RESOLVED)

        who = _retry_call(HF_API.whoami)
        print("[HF AUTH] account:", who.get("name"))

        # IMPORTANT: dataset_info is a type-specific verification.
        # If this succeeds, we know we are addressing a Dataset repo,
        # not a model repo.
        info = _retry_call(
            HF_API.dataset_info,
            repo_id=HF_REPO_ID_RESOLVED,
            revision="main",
        )

        canonical_id = getattr(info, "id", None) or HF_REPO_ID_RESOLVED
        if canonical_id != HF_REPO_ID_RESOLVED:
            print(
                f"[HF DATASET] Hub canonicalized repo id: "
                f"{HF_REPO_ID_RESOLVED} -> {canonical_id}"
            )
            HF_REPO_ID_RESOLVED = canonical_id
            os.environ["HF_REPO_ID"] = canonical_id
            os.environ["CIDM_HF_DATASET_REPO"] = canonical_id

        print(
            f"[HF DATASET VERIFIED] "
            f"https://huggingface.co/datasets/{HF_REPO_ID_RESOLVED}"
        )

        HF_ENABLED = True
        restored_state = restore_progress_from_hf()

        print(
            f"[HF RESUME READY] completed="
            f"{restored_state['completed']}/{restored_state['total_expected']}; "
            f"checkpoints={restored_state['latest_checkpoints']}"
        )

    except Exception as e:
        HF_ENABLED = False
        if HF_ALLOW_LOCAL_ONLY:
            print(f"[HF DATASET WARNING] {repr(e)}")
            print(
                "CIDM_HF_ALLOW_LOCAL_ONLY=1, so local execution may continue."
            )
        else:
            raise RuntimeError(
                "Hugging Face Dataset restore failed. Training is blocked "
                "rather than silently restarting from zero. "
                f"Target dataset repo: {HF_REPO_ID_RESOLVED}. "
                f"Underlying error: {repr(e)}"
            ) from e


In [ ]:

# ============================================================
# 3. Official data / cache
# ============================================================
TASKS={
    "burgers":{
        "filename":"1D_Burgers_Sols_Nu0.01.hdf5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/281363",
        "md5":"e6d9a4f62baf9a29121a816b919e2770",
    },
    "diff_react_2d":{
        "filename":"2D_diff-react_NA_NA.h5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/133017",
        "md5":"b8d0b86064193195ddc30c33be5dc949",
    },
    "swe_2d":{
        "filename":"2D_rdb_NA_NA.h5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/133021",
        "md5":"75d838c47aa410694bdc912ea7f22282",
    },
}

def choose_root(env,defaults,required=None):
    cands=[]
    if os.getenv(env): cands.append(Path(os.environ[env]))
    cands += [Path(x) for x in defaults]
    if required:
        for c in cands:
            if all((c/f).exists() for f in required):
                return c
    for c in cands:
        if c.exists(): return c
    return cands[0]

DATA_ROOT=choose_root(
    "CIDM_PDEBENCH_DATA_ROOT",
    ["/content/CIDM_PDEBench_Data",str(ROOT/"CIDM_PDEBench_Data")],
    [x["filename"] for x in TASKS.values()]
)
CACHE_ROOT=choose_root(
    "CIDM_PDEBENCH_A1R2_CACHE_ROOT",
    ["/content/CIDM_PDEBench_A1R2_Cache",str(ROOT/"CIDM_PDEBench_A1R2_Cache")]
)
DATA_ROOT.mkdir(parents=True,exist_ok=True)
CACHE_ROOT.mkdir(parents=True,exist_ok=True)

def resume_download(url,path,expected_md5,chunk=8*1024*1024):
    path=Path(path)
    if path.exists() and md5_file(path)==expected_md5:
        print("[verified data]",path.name)
        return
    if path.exists(): path.unlink()
    part=path.with_suffix(path.suffix+".part")
    pos=part.stat().st_size if part.exists() else 0
    headers={"Range":f"bytes={pos}-"} if pos else {}
    with requests.get(url,stream=True,headers=headers,timeout=(30,300)) as r:
        if pos and r.status_code==200:
            part.unlink(missing_ok=True); pos=0
        r.raise_for_status()
        mode="ab" if pos and r.status_code==206 else "wb"
        with open(part,mode) as f:
            for b in r.iter_content(chunk_size=chunk):
                if b: f.write(b)
    part.replace(path)
    got=md5_file(path)
    if got!=expected_md5:
        raise RuntimeError(f"MD5 mismatch {path.name}: {got}")

for task,m in TASKS.items():
    p=DATA_ROOT/m["filename"]
    resume_download(m["url"],p,m["md5"])
    m["path"]=str(p)

def inspect_h5(path):
    with h5py.File(path,"r") as f:
        if "tensor" in f:
            return {"schema":"tensor","n":f["tensor"].shape[0]}
        groups=[k for k in sorted(f.keys()) if isinstance(f[k],h5py.Group) and "data" in f[k]]
        if groups:
            return {"schema":"grouped","n":len(groups)}
        raise RuntimeError(f"unsupported HDF5 schema: {path}")

def read_traj(path,idx):
    with h5py.File(path,"r") as f:
        if "tensor" in f:
            a=np.asarray(f["tensor"][idx],dtype=np.float32)
            if a.ndim==2: a=a[:,None,:]
            elif a.ndim==3: a=a[:,None,:,:]
            else: raise RuntimeError(a.shape)
        else:
            groups=[k for k in sorted(f.keys()) if isinstance(f[k],h5py.Group) and "data" in f[k]]
            a=np.asarray(f[groups[idx]]["data"],dtype=np.float32)
            if a.ndim==4: a=np.moveaxis(a,-1,1)
            elif a.ndim==3: a=a[:,None,:,:]
            else: raise RuntimeError(a.shape)
    return torch.from_numpy(a)

def even_pick(pool,n):
    pool=np.asarray(pool,dtype=int)
    n=min(n,len(pool))
    if n<=0: return []
    return pool[np.linspace(0,len(pool)-1,n,dtype=int)].tolist()

def split_ids(N):
    nt=max(1,int(.10*N))
    test_pool=np.arange(0,nt)
    trainval=np.arange(nt,N)
    nv=max(1,int(.10*N))
    val_pool=trainval[-nv:]
    train_pool=trainval[:-nv]
    return {
        "train":even_pick(train_pool,N_TRAIN_TRAJ),
        "val":even_pick(val_pool,N_VAL_TRAJ),
        "test":even_pick(test_pool,N_TEST_TRAJ),
    }

def cache_key(task,m):
    return (
        f"{task}_L{HISTORY}_K{HORIZON}"
        f"_tr{N_TRAIN_TRAJ}_va{N_VAL_TRAJ}_te{N_TEST_TRAJ}"
        f"_w{WINDOWS_PER_TRAJ}.pt"
    )

def build_cache(task,m):
    cp=CACHE_ROOT/cache_key(task,m)
    if cp.exists():
        try:
            c=torch.load(cp,map_location="cpu",weights_only=False)
            if (
                c.get("source_md5")==m["md5"]
                and c.get("history")==HISTORY
                and c.get("horizon")==HORIZON
            ):
                print("[reuse cache]",cp.name)
                return c
        except Exception:
            pass

    info=inspect_h5(m["path"])
    ids=split_ids(info["n"])
    out={
        "source_md5":m["md5"],
        "history":HISTORY,
        "horizon":HORIZON,
        "selected_trajectory_ids":ids,
        "splits":{},
    }

    for split,idxs in ids.items():
        HH=[]; YY=[]; WM=[]
        for idx in idxs:
            tr=read_traj(m["path"],idx)
            mx=tr.shape[0]-HISTORY-HORIZON
            if mx<0: raise RuntimeError(f"{task}: trajectory shorter than L+K")
            starts=np.unique(np.linspace(0,mx,WINDOWS_PER_TRAJ,dtype=int))
            for st in starts:
                HH.append(tr[st:st+HISTORY])
                YY.append(tr[st+HISTORY:st+HISTORY+HORIZON])
                WM.append((int(idx),int(st)))
        out["splits"][split]={
            "history":torch.stack(HH).contiguous(),
            "target":torch.stack(YY).contiguous(),
            "window_meta":WM,
        }

    # Per-channel training normalization statistics from history + target.
    h=out["splits"]["train"]["history"].double()
    y=out["splits"]["train"]["target"].double()
    C=h.shape[2]
    sum_c=torch.zeros(C,dtype=torch.float64)
    sq_c=torch.zeros(C,dtype=torch.float64)
    count=0
    for q in [h,y]:
        reduce_dims=(0,1)+tuple(range(3,q.ndim))
        sum_c += q.sum(dim=reduce_dims)
        sq_c += (q*q).sum(dim=reduce_dims)
        n=1
        for d in reduce_dims: n*=q.shape[d]
        count += n
    mean=sum_c/count
    var=(sq_c/count-mean*mean).clamp_min(1e-12)
    std=torch.sqrt(var)
    out["norm"]={"mean":mean.float(),"std":std.float()}

    torch.save(out,cp)
    print("[built cache]",cp)
    return out

CACHES={t:build_cache(t,m) for t,m in TASKS.items()}

class ChannelNormalizer:
    def __init__(self,mean,std):
        self.mean=mean.float()
        self.std=std.float().clamp_min(1e-6)
    def _shape(self,x):
        return [1]*(x.ndim-2)+[1]  # unused
    def norm_history(self,x):
        # [B,L,C,*S]
        shape=[1,1,-1]+[1]*(x.ndim-3)
        return (x-self.mean.view(*shape).to(x.device))/self.std.view(*shape).to(x.device)
    def norm_target(self,x):
        # [B,K,C,*S]
        shape=[1,1,-1]+[1]*(x.ndim-3)
        return (x-self.mean.view(*shape).to(x.device))/self.std.view(*shape).to(x.device)
    def denorm_target(self,x):
        shape=[1,1,-1]+[1]*(x.ndim-3)
        return x*self.std.view(*shape).to(x.device)+self.mean.view(*shape).to(x.device)

NORMALIZERS={
    t:ChannelNormalizer(c["norm"]["mean"],c["norm"]["std"])
    for t,c in CACHES.items()
}

class FrozenWindows(Dataset):
    def __init__(self,c,split,norm):
        self.h=c["splits"][split]["history"].float()
        self.y=c["splits"][split]["target"].float()
        self.norm=norm
    def __len__(self): return len(self.h)
    def __getitem__(self,i):
        h=self.h[i]
        y=self.y[i]
        # Normalize without batch dimensions.
        mh=self.norm.mean.view(1,-1,*([1]*(h.ndim-2)))
        sh=self.norm.std.view(1,-1,*([1]*(h.ndim-2)))
        return (h-mh)/sh,(y-mh)/sh

def loaders_for(task,batch,seed):
    c=CACHES[task]; norm=NORMALIZERS[task]
    g=torch.Generator().manual_seed(seed)
    return (
        DataLoader(FrozenWindows(c,"train",norm),batch_size=batch,shuffle=True,
                   generator=g,num_workers=0,drop_last=True,pin_memory=True),
        DataLoader(FrozenWindows(c,"val",norm),batch_size=batch,shuffle=False,
                   num_workers=0,pin_memory=True),
        DataLoader(FrozenWindows(c,"test",norm),batch_size=batch,shuffle=False,
                   num_workers=0,pin_memory=True),
    )

def task_spec(task):
    h=CACHES[task]["splits"]["train"]["history"]
    return h.ndim-3,h.shape[2],tuple(h.shape[3:])

manifest=[]
for t in TASKS:
    d,c,s=task_spec(t)
    cc=CACHES[t]
    manifest.append({
        "task":t,"dim":d,"channels":c,"spatial":list(s),
        "train_windows":len(cc["splits"]["train"]["history"]),
        "val_windows":len(cc["splits"]["val"]["history"]),
        "test_windows":len(cc["splits"]["test"]["history"]),
        "norm_mean":cc["norm"]["mean"].tolist(),
        "norm_std":cc["norm"]["std"].tolist(),
    })
display(pd.DataFrame(manifest))
save_json(manifest,OUT/"data_window_manifest.json")
REPORT["data_manifest"]=manifest



# 4. Official / Frozen I/O wrappers

Wrapper 只允许做：
- tensor layout mapping；
- regular-grid coordinate construction；
- autoregressive history update；
- CIDM-Complete 的 1D `H=1` lift/squeeze。

禁止：
- 修改官方 baseline 内部 layer；
- 修改冻结 CIDM Core；
- 修改冻结 CIDM-Complete 内部公式；
- 给 Complete 额外 forcing/static information；
- 使用未来 target 参与 inference。


In [ ]:

# ============================================================
# 4. Official wrappers
# ============================================================
def normalized_grid(spatial,device):
    if len(spatial)==1:
        return torch.linspace(0,1,spatial[0],device=device)[:,None]
    x=torch.linspace(0,1,spatial[0],device=device)
    y=torch.linspace(0,1,spatial[1],device=device)
    X,Y=torch.meshgrid(x,y,indexing="ij")
    return torch.stack([X,Y],-1)

class CIDMWrapper(nn.Module):
    def __init__(self,dim,c,L,K,D,H):
        super().__init__()
        self.model=build_cidm_foundation_core(
            spatial_dim=dim,channels=c,hist_len=L,horizon=K,
            latent_dim=D,hidden=H
        )
    def rollout(self,h,K):
        return self.model(h,horizon=K)["pred"]


class CIDMCompleteWrapper(nn.Module):
    """
    Exact archived CIDM_BER topology used as CIDM-Complete.
    For 1D tasks we lift X -> [1,X] and use the same archived Conv2D model.
    No external forcing/port is supplied in the PDEBench headline panel.
    """
    def __init__(self,dim,c,L,K,D,H,modes1,modes2):
        super().__init__()
        self.dim=dim
        self.config=CIDMCompleteConfig(
            channels=c,
            hist_len=L,
            horizon=K,
            latent_dim=D,
            hidden=H,
            port_channels=1,
            use_phase_encoder=True,
            use_time_encoder=True,
            use_hidden_reservoir=True,
            use_flux=True,
            use_reverse_diagnostics=True,
            soc_modes1=modes1,
            soc_modes2=modes2,
            ber_add_scale=0.05,
            ber_enabled=True,
            soc_enabled=True,
        )
        self.model=CIDMCompleteArchived(self.config)

    def _lift(self,h):
        return h.unsqueeze(-2) if self.dim==1 else h

    def rollout(self,h,K):
        hh=self._lift(h)
        out=self.model(hh,horizon=K,port=None)
        p=out["pred"]
        return p.squeeze(-2) if self.dim==1 else p

    @torch.no_grad()
    def architecture_diagnostics(self,h,K):
        hh=self._lift(h)
        out=self.model(hh,horizon=K,port=None)
        d=out["diagnostics"]
        return {
            "latent_energy_drift":float(d["energy_drift"].detach()),
            "soc_strength":float(d["soc_strength"].detach()),
            "ber_strength":float(d["ber_strength"].detach()),
        }


class PDEBenchFNOWrapper(nn.Module):
    def __init__(self,dim,c,L,width,modes):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        if dim==1:
            self.model=PDE_FNO1d(c,modes=modes,width=width,initial_step=L)
        else:
            self.model=PDE_FNO2d(c,modes1=modes,modes2=modes,width=width,initial_step=L)
    def one(self,h):
        B,L,C,*S=h.shape
        if self.dim==1:
            inp=h.permute(0,3,1,2).reshape(B,S[0],L*C)
        else:
            inp=h.permute(0,3,4,1,2).reshape(B,S[0],S[1],L*C)
        grid=normalized_grid(S,h.device)[None].expand(B,*([-1]*self.dim),-1)
        y=self.model(inp,grid).squeeze(-2)
        if self.dim==1: return y.permute(0,2,1).contiguous()
        return y.permute(0,3,1,2).contiguous()
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            y=self.one(h); ps.append(y)
            h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class PDEBenchUNetWrapper(nn.Module):
    def __init__(self,dim,c,L,features):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        self.model=(PDE_UNet1d if dim==1 else PDE_UNet2d)(
            in_channels=c*L,out_channels=c,init_features=features
        )
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            B,L,C,*S=h.shape
            y=self.model(h.reshape(B,L*C,*S))
            ps.append(y); h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class NeuralOpFNOWrapper(nn.Module):
    def __init__(self,dim,c,L,width,modes):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        self.model=NeuralOpFNO(
            n_modes=tuple([modes]*dim),
            in_channels=c*L,out_channels=c,
            hidden_channels=width,n_layers=4,
            positional_embedding="grid",
            fno_block_precision="full",
        )
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            B,L,C,*S=h.shape
            y=self.model(h.reshape(B,L*C,*S))
            ps.append(y); h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class TransolverWrapper(nn.Module):
    def __init__(self,dim,c,L,S,n_hidden,n_layers,n_head,slice_num):
        super().__init__(); self.dim=dim; self.c=c; self.L=L; self.S=S
        kw=dict(
            space_dim=dim,n_layers=n_layers,n_hidden=n_hidden,dropout=0.0,
            n_head=n_head,Time_Input=False,act="gelu",mlp_ratio=1,
            fun_dim=c*L,out_dim=c,slice_num=slice_num,unified_pos=False
        )
        if dim==1:
            self.model=TransolverIrregular(**kw)
        else:
            self.model=Transolver2D(**kw,H=S[0],W=S[1])
    def one(self,h):
        B,L,C,*S=h.shape
        g=normalized_grid(S,h.device).reshape(-1,self.dim)[None].expand(B,-1,-1)
        if self.dim==1:
            fx=h.permute(0,3,1,2).reshape(B,-1,L*C)
        else:
            fx=h.permute(0,3,4,1,2).reshape(B,-1,L*C)
        y=self.model(g,fx)
        if self.dim==1:
            return y.reshape(B,S[0],C).permute(0,2,1).contiguous()
        return y.reshape(B,S[0],S[1],C).permute(0,3,1,2).contiguous()
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            y=self.one(h); ps.append(y)
            h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)



# 5. Frozen 0.5M parameter configurations

A1-R2 已经完成参数匹配。本轮属于 publication confirmation，
所以**禁止重新搜索模型宽度**。

直接冻结上一轮通过的 18 个 task × model 配置，然后重新实例化计数验证：

\[
500K\pm5\%
\]

这可以避免在看到上一轮性能之后继续调整 architecture knobs。


In [ ]:

# ============================================================
# 5. Frozen 0.5M configs + parameter audit
# ============================================================
FROZEN_MATCH={
    "burgers":{
        "CIDM-Foundation-v1":{"latent_dim":104,"hidden":160},
        "CIDM-Complete":{"latent_dim":42,"hidden":72,"soc_modes1":1,"soc_modes2":4},
        "PDEBench-FNO":{"width":110,"modes":9},
        "PDEBench-U-Net":{"features":14},
        "NeuralOperator-FNO":{"width":94,"modes":20},
        "Transolver":{"n_hidden":132,"n_layers":5,"n_head":4,"slice_num":32},
    },
    "diff_react_2d":{
        "CIDM-Foundation-v1":{"latent_dim":64,"hidden":96},
        "CIDM-Complete":{"latent_dim":20,"hidden":92,"soc_modes1":2,"soc_modes2":14},
        "PDEBench-FNO":{"width":18,"modes":14},
        "PDEBench-U-Net":{"features":8},
        "NeuralOperator-FNO":{"width":68,"modes":6},
        "Transolver":{"n_hidden":68,"n_layers":5,"n_head":4,"slice_num":8},
    },
    "swe_2d":{
        "CIDM-Foundation-v1":{"latent_dim":36,"hidden":108},
        "CIDM-Complete":{"latent_dim":22,"hidden":84,"soc_modes1":4,"soc_modes2":12},
        "PDEBench-FNO":{"width":18,"modes":14},
        "PDEBench-U-Net":{"features":8},
        "NeuralOperator-FNO":{"width":68,"modes":6},
        "Transolver":{"n_hidden":68,"n_layers":5,"n_head":4,"slice_num":8},
    },
}

def build_by_config(task,name,cfg):
    dim,c,S=task_spec(task)
    if name=="CIDM-Foundation-v1":
        return CIDMWrapper(
            dim,c,HISTORY,HORIZON,cfg["latent_dim"],cfg["hidden"]
        )
    if name=="CIDM-Complete":
        return CIDMCompleteWrapper(
            dim,c,HISTORY,HORIZON,
            cfg["latent_dim"],cfg["hidden"],
            cfg["soc_modes1"],cfg["soc_modes2"]
        )
    if name=="PDEBench-FNO":
        return PDEBenchFNOWrapper(dim,c,HISTORY,cfg["width"],cfg["modes"])
    if name=="PDEBench-U-Net":
        return PDEBenchUNetWrapper(dim,c,HISTORY,cfg["features"])
    if name=="NeuralOperator-FNO":
        return NeuralOpFNOWrapper(dim,c,HISTORY,cfg["width"],cfg["modes"])
    if name=="Transolver":
        return TransolverWrapper(
            dim,c,HISTORY,S,cfg["n_hidden"],cfg["n_layers"],
            cfg["n_head"],cfg["slice_num"]
        )
    raise KeyError(name)

MATCH={}
rows=[]
for task in TASKS:
    MATCH[task]={}
    for name in MODELS:
        cfg=FROZEN_MATCH[task][name]
        m=build_by_config(task,name,cfg)
        p=count_params(m)
        rel=abs(p-TARGET_PARAMS)/TARGET_PARAMS
        MATCH[task][name]={
            "params":p,"rel_error":rel,"ok":rel<=PARAM_TOL,"config":cfg
        }
        rows.append({
            "task":task,"model":name,"params":p,
            "rel_error":rel,"ok":rel<=PARAM_TOL,
            "config":json.dumps(cfg),
        })
        print(task,name,p,f"{100*rel:.3f}%","PASS" if rel<=PARAM_TOL else "FAIL")
        del m
        gc.collect()

match_df=pd.DataFrame(rows)
match_df.to_csv(OUT/"parameter_matching_frozen.csv",index=False)
display(match_df)

if not match_df["ok"].all():
    raise RuntimeError("Frozen parameter audit failed.")

save_json(MATCH,OUT/"parameter_matching_frozen.json")
REPORT["parameter_matching"]=rows



# 6. Pre-flight gate

正式训练前：

\[
6\ models \times 3\ tasks = 18
\]

个组合必须全部完成：

```text
real normalized batch
→ 6-step rollout
→ shape audit
→ finite audit
→ backward
→ gradient finite
→ one optimizer step
→ checkpoint save/reload
→ inference finite
```

CIDM-Complete 额外验证：
- 1D H=1 lift/squeeze；
- SOC/BER forward；
- zero-port contract；
- 完整 archive checkpoint reload。

任何失败都阻断正式比较。


In [ ]:

# ============================================================
# 6. Pre-flight
# ============================================================
PREFLIGHT=[]
for task in TASKS:
    dim,c,S=task_spec(task)
    batch=BATCH_1D if dim==1 else BATCH_2D
    tr,va,te=loaders_for(task,batch,DEV_SEED)
    h,y=next(iter(tr)); h,y=h.to(DEVICE),y.to(DEVICE)

    for name in MODELS:
        seed_all(DEV_SEED)
        cfg=MATCH[task][name]["config"]
        m=build_by_config(task,name,cfg).to(DEVICE).float()
        opt=torch.optim.AdamW(m.parameters(),lr=1e-3,weight_decay=WEIGHT_DECAY)
        ok=False; err=""
        try:
            m.train(); opt.zero_grad(set_to_none=True)
            p=m.rollout(h,HORIZON)
            if p.shape!=y.shape:
                raise RuntimeError(f"shape {p.shape} != {y.shape}")
            if not torch.isfinite(p).all():
                raise FloatingPointError("non-finite forward")
            loss=F.mse_loss(p,y)
            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite loss")
            loss.backward()
            bad=[
                n for n,q in m.named_parameters()
                if q.grad is not None and not torch.isfinite(q.grad).all()
            ]
            if bad:
                raise FloatingPointError(f"non-finite gradients: {bad[:8]}")
            torch.nn.utils.clip_grad_norm_(m.parameters(),1.0)
            opt.step()
            if not all(torch.isfinite(q).all() for q in m.parameters()):
                raise FloatingPointError("non-finite params")

            sd=portable_state_dict(m)
            temp=CKPT/f"preflight__{task}__{name}.pt"
            torch.save(sd,temp)

            fresh=build_by_config(task,name,cfg).to(DEVICE).float()
            load_portable_state_dict(
                fresh,torch.load(temp,map_location="cpu",weights_only=False)
            )
            fresh.eval()
            with torch.no_grad():
                pp=fresh.rollout(h,HORIZON)
            if not torch.isfinite(pp).all():
                raise FloatingPointError("non-finite after reload")
            ok=True
        except Exception as e:
            err=repr(e)

        row={"task":task,"model":name,"pass":ok,"error":err}
        PREFLIGHT.append(row); print(row)
        del m
        if "fresh" in locals(): del fresh
        torch.cuda.empty_cache(); gc.collect()

pre_df=pd.DataFrame(PREFLIGHT)
pre_df.to_csv(OUT/"preflight.csv",index=False)
display(pre_df)
if not pre_df["pass"].all():
    raise RuntimeError("Pre-flight failed. Formal training is blocked.")
REPORT["preflight"]=PREFLIGHT



# 7. Publication metrics

所有 headline prediction metrics 仍在反归一化后的原始变量空间计算。

### 预测
- RMSE
- MAE
- Relative L2
- lead-wise RMSE
- rollout growth

### 修复后的频谱指标
旧版 relative band error 在目标 band energy 接近 0 时会爆炸，本轮不再使用。

输出：

1. `band_fraction_abs_error_{low,mid,high}`
2. `logspec_rmse_{low,mid,high}`

### 效率
- actual updates to convergence
- best validation step
- GPU seconds
- 6-step latency
- throughput
- peak VRAM


In [ ]:

# ============================================================
# 7. Publication metrics
# ============================================================
def _spectral_masks(shape,device):
    if len(shape)==1:
        f=torch.fft.rfftfreq(shape[0],device=device)
        q=f/f.max().clamp_min(1e-12)
    else:
        fy=torch.fft.fftfreq(shape[-2],device=device)[:,None]
        fx=torch.fft.rfftfreq(shape[-1],device=device)[None,:]
        r=torch.sqrt(fx*fx+fy*fy)
        q=r/r.max().clamp_min(1e-12)
    return {
        "low":q<=1/3,
        "mid":(q>1/3)&(q<=2/3),
        "high":q>2/3,
    }

def _publication_spectral_metrics(pred,target):
    # pred/target [B,K,C,*S], original physical units.
    dim=pred.ndim-3
    if dim==1:
        P=torch.fft.rfft(pred.float(),dim=-1,norm="ortho").abs()
        T=torch.fft.rfft(target.float(),dim=-1,norm="ortho").abs()
        masks=_spectral_masks((pred.shape[-1],),pred.device)
    else:
        P=torch.fft.rfft2(pred.float(),dim=(-2,-1),norm="ortho").abs()
        T=torch.fft.rfft2(target.float(),dim=(-2,-1),norm="ortho").abs()
        masks=_spectral_masks((pred.shape[-2],pred.shape[-1]),pred.device)

    P2=P.square()
    T2=T.square()
    reduce_freq=(-1,) if dim==1 else (-2,-1)
    totalP=P2.sum(dim=reduce_freq).clamp_min(1e-12)
    totalT=T2.sum(dim=reduce_freq).clamp_min(1e-12)

    out={}
    logP=torch.log1p(P)
    logT=torch.log1p(T)

    for band,m in masks.items():
        # Band energy fraction: stable even if true band energy is almost zero.
        bp=P2[...,m].sum(dim=-1)
        bt=T2[...,m].sum(dim=-1)
        fp=bp/totalP
        ft=bt/totalT
        out[f"band_fraction_abs_error_{band}"]=float((fp-ft).abs().mean())

        # Log-spectrum RMSE in that band.
        d=(logP[...,m]-logT[...,m]).square().mean().sqrt()
        out[f"logspec_rmse_{band}"]=float(d)

        # Store mean true fraction for interpretability.
        out[f"true_band_fraction_{band}"]=float(ft.mean())

    return out

@torch.no_grad()
def evaluate_model(model,loader,task,max_batches=None,save_examples=False,example_tag=None):
    model.eval()
    norm=NORMALIZERS[task]
    mse=[]; mae=[]; rel=[]; leads=[]; specs=[]
    n_samples=0
    first_example=None

    torch.cuda.synchronize()
    t0=time.perf_counter()

    for bi,(h,y_n) in enumerate(loader):
        if max_batches is not None and bi>=max_batches:
            break
        h=h.to(DEVICE,non_blocking=True)
        y_n=y_n.to(DEVICE,non_blocking=True)

        p_n=model.rollout(h,HORIZON)
        p=norm.denorm_target(p_n)
        y=norm.denorm_target(y_n)
        e=p-y

        mse.append(float((e**2).mean()))
        mae.append(float(e.abs().mean()))
        rel.append(float(
            torch.linalg.vector_norm(e.reshape(e.shape[0],-1),dim=1).div(
                torch.linalg.vector_norm(y.reshape(y.shape[0],-1),dim=1).clamp_min(1e-8)
            ).mean()
        ))
        leads.append(torch.sqrt((e**2).flatten(2).mean(-1)).mean(0).cpu())
        specs.append(_publication_spectral_metrics(p,y))
        n_samples += h.shape[0]

        if first_example is None:
            # save one deterministic example in original units
            hh=norm.denorm_target(h[:, -1:, ...]).squeeze(1)
            first_example={
                "last_input":hh[0].detach().cpu(),
                "target":y[0].detach().cpu(),
                "pred":p[0].detach().cpu(),
            }

    torch.cuda.synchronize()
    elapsed=time.perf_counter()-t0

    le=torch.stack(leads).mean(0)
    out={
        "rmse":math.sqrt(np.mean(mse)),
        "mae":float(np.mean(mae)),
        "relative_l2":float(np.mean(rel)),
        "lead_rmse":[float(x) for x in le],
        "rollout_growth":float(le[-1]/le[0].clamp_min(1e-8)),
        "eval_throughput_windows_per_s":float(n_samples/max(elapsed,1e-9)),
    }

    for k in specs[0].keys():
        out[k]=float(np.mean([x[k] for x in specs]))

    if save_examples and first_example is not None and example_tag:
        torch.save(first_example,PRED/f"{example_tag}.pt")

    return out

@torch.no_grad()
def measure_latency(model,loader,repeats=50,warmup=15):
    model.eval()
    h,_=next(iter(loader))
    h=h[:1].to(DEVICE)

    for _ in range(warmup):
        _=model.rollout(h,HORIZON)
    torch.cuda.synchronize()

    ts=[]
    for _ in range(repeats):
        torch.cuda.synchronize()
        t0=time.perf_counter()
        _=model.rollout(h,HORIZON)
        torch.cuda.synchronize()
        ts.append((time.perf_counter()-t0)*1000)

    return {
        "latency_ms_median":float(np.median(ts)),
        "latency_ms_mean":float(np.mean(ts)),
        "latency_ms_p95":float(np.percentile(ts,95)),
    }

@torch.no_grad()
def architecture_specific_diagnostics(model,loader,name,max_batches=4):
    if name not in {"CIDM-Foundation-v1","CIDM-Complete"}:
        return {}

    rows=[]
    for bi,(h,y) in enumerate(loader):
        if bi>=max_batches: break
        h=h.to(DEVICE)

        if name=="CIDM-Foundation-v1":
            out=model.model(h,horizon=HORIZON)
            d=out["diagnostics"]
            rows.append({
                "latent_energy_drift":float(d["latent_energy_drift"].detach()),
                "local_strength":float(d["local_strength"].detach()),
                "step_size":float(d["step_size"].detach()),
            })
        else:
            rows.append(model.architecture_diagnostics(h,HORIZON))

    keys=sorted({k for r in rows for k in r})
    return {k:float(np.mean([r[k] for r in rows if k in r])) for k in keys}



# 8. Frozen learning-rate contract

A1-R2 已经通过完全相同的 4-LR 搜索预算选出每个架构的一个 global LR。

Publication confirmation **不得重新调参**。

本轮直接冻结上一轮结果，并写入 protocol manifest。


In [ ]:

# ============================================================
# 8. Frozen learning-rate audit
# ============================================================
LR_SELECTED={
    "CIDM-Foundation-v1":1e-4,
    "CIDM-Complete":1e-3,
    "PDEBench-FNO":3e-3,
    "PDEBench-U-Net":3e-3,
    "NeuralOperator-FNO":3e-3,
    "Transolver":3e-4,
}
assert set(LR_SELECTED)==set(MODELS)
save_json(LR_SELECTED,OUT/"selected_lr_frozen.json")
REPORT["selected_lr"]=LR_SELECTED

lr_rows=[{"model":m,"lr":LR_SELECTED[m],"source":"A1-R2 equal-budget global-LR qualification"} for m in MODELS]
pd.DataFrame(lr_rows).to_csv(OUT/"selected_lr_frozen.csv",index=False)
display(pd.DataFrame(lr_rows))



# 9. Ten-seed convergence-confirmation benchmark

正式规模：

\[
6\ models \times 3\ tasks \times 10\ seeds
=
180\ runs
\]

本轮不是 fixed-step ranking，而是统一 convergence contract：

```text
MAX_UPDATES      = 4000
MIN_UPDATES      = 1500
EVAL_EVERY       = 100
PATIENCE_EVALS   = 8
REL_MIN_DELTA    = 0.1%
```

### Early stop 的公平定义

在至少 `MIN_UPDATES` 后，如果连续 `PATIENCE_EVALS` 次 validation evaluation
都没有将当前“收敛参考 best”改善至少 `REL_MIN_DELTA`，则停止。

注意：
- 最终模型仍加载**raw lowest validation RMSE checkpoint**；
- 收敛判据单独使用 relative threshold，防止数值抖动不断重置 patience；
- 所有模型完全同一规则。

### Exact resume

训练 batch 不依赖 DataLoader 当前 iterator 状态。
每一个 `(task, seed)` 都预先生成 deterministic step-wise batch plan：

```text
step 1 -> fixed indices
step 2 -> fixed indices
...
step MAX_UPDATES -> fixed indices
```

因此 Colab 中断后从 checkpoint 恢复不会改变后续样本顺序。


In [ ]:

# ============================================================
# 9. Exact deterministic convergence training
#    + cross-Colab HF checkpoint/run synchronization
# ============================================================
def normalized_split_tensors(task,split):
    c=CACHES[task]
    norm=NORMALIZERS[task]
    h=c["splits"][split]["history"].float()
    y=c["splits"][split]["target"].float()

    mh=norm.mean.view(1,1,-1,*([1]*(h.ndim-3)))
    sh=norm.std.view(1,1,-1,*([1]*(h.ndim-3)))
    return ((h-mh)/sh).contiguous(),((y-mh)/sh).contiguous()

TRAIN_TENSORS={}
for task in TASKS:
    TRAIN_TENSORS[task]=normalized_split_tensors(task,"train")

def deterministic_batch_plan(n,batch,seed,steps):
    """
    Exact list of sample-index arrays for every optimizer step.
    Same task/seed => same batches for every architecture and after resume.
    """
    rng=np.random.default_rng(seed)
    plan=[]
    while len(plan)<steps:
        perm=rng.permutation(n)
        usable=(len(perm)//batch)*batch
        perm=perm[:usable]
        if usable==0:
            raise RuntimeError(f"dataset size {n} smaller than batch {batch}")
        for j in range(0,usable,batch):
            plan.append(perm[j:j+batch].astype(np.int64))
            if len(plan)>=steps:
                break
    return plan

def train_to_convergence(
    model,task,va,lr,seed,tag,latest_path
):
    model=model.to(DEVICE).float()
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,T_max=MAX_UPDATES,eta_min=lr*0.1
    )

    train_h,train_y=TRAIN_TENSORS[task]
    batch=BATCH_1D if task_spec(task)[0]==1 else BATCH_2D
    plan=deterministic_batch_plan(len(train_h),batch,seed,MAX_UPDATES)

    start_step=0
    raw_best=float("inf")
    raw_best_step=0
    best_sd=None
    convergence_ref=float("inf")
    bad_evals=0
    history=[]

    # Accumulated wall/GPU-session training time from new-format checkpoints.
    elapsed_before=0.0
    legacy_elapsed_unknown=False

    if latest_path.exists():
        ck=torch.load(latest_path,map_location="cpu",weights_only=False)
        load_portable_state_dict(model,ck["model"])
        opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"])

        start_step=int(ck["step"])
        raw_best=float(ck["raw_best"])
        raw_best_step=int(ck["raw_best_step"])
        best_sd=ck.get("best_model")
        convergence_ref=float(ck.get("convergence_ref",raw_best))
        bad_evals=int(ck.get("bad_evals",0))
        history=list(ck.get("history",[]))

        if "elapsed_train_seconds" in ck:
            elapsed_before=float(ck["elapsed_train_seconds"])
        else:
            # User's already-uploaded legacy checkpoint can still be resumed
            # exactly in optimization state and sample order. Only earlier
            # wall-time is unavailable for that one interrupted segment.
            legacy_elapsed_unknown = start_step > 0

        print(
            f"[exact HF/local resume] {tag} step={start_step}, "
            f"raw_best={raw_best:.6g}@{raw_best_step}"
        )
        if legacy_elapsed_unknown:
            print(
                "[timing note] legacy checkpoint has no cumulative elapsed time; "
                "optimization resume is exact, but prior-session training seconds "
                "cannot be reconstructed."
            )

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    session_t0=time.perf_counter()
    stop_reason="MAX_UPDATES"

    for s in range(start_step+1,MAX_UPDATES+1):
        idx=torch.as_tensor(plan[s-1],dtype=torch.long)
        h=train_h.index_select(0,idx).to(DEVICE,non_blocking=True)
        y=train_y.index_select(0,idx).to(DEVICE,non_blocking=True)

        model.train()
        opt.zero_grad(set_to_none=True)
        p=model.rollout(h,HORIZON)
        loss=F.mse_loss(p,y)

        if not torch.isfinite(loss):
            raise FloatingPointError(f"{tag}: non-finite loss at step {s}")

        loss.backward()

        bad=[
            n for n,q in model.named_parameters()
            if q.grad is not None and not torch.isfinite(q.grad).all()
        ]
        if bad:
            raise FloatingPointError(f"{tag}: non-finite gradients {bad[:8]}")

        grad_norm=float(torch.nn.utils.clip_grad_norm_(model.parameters(),1.0))
        opt.step()
        sched.step()

        should_stop=False

        if s==1 or s%EVAL_EVERY==0 or s==MAX_UPDATES:
            vr=evaluate_model(model,va,task)["rmse"]

            # Raw best checkpoint.
            if vr<raw_best:
                raw_best=vr
                raw_best_step=s
                best_sd=portable_state_dict(model)

            # Thresholded convergence reference.
            if not math.isfinite(convergence_ref):
                convergence_ref=vr
                bad_evals=0
            else:
                required=convergence_ref*(1.0-REL_MIN_DELTA)
                if vr<required:
                    convergence_ref=vr
                    bad_evals=0
                elif s>=MIN_UPDATES:
                    bad_evals+=1

            row={
                "step":s,
                "train_normalized_mse":float(loss.detach()),
                "val_rmse_original":float(vr),
                "raw_best_val_rmse":float(raw_best),
                "raw_best_step":int(raw_best_step),
                "convergence_ref":float(convergence_ref),
                "bad_evals":int(bad_evals),
                "lr":float(opt.param_groups[0]["lr"]),
                "grad_norm_preclip":grad_norm,
            }
            history.append(row)

            print(
                f"[{tag}] {s}/{MAX_UPDATES} "
                f"val={vr:.6g} best={raw_best:.6g}@{raw_best_step} "
                f"bad={bad_evals}/{PATIENCE_EVALS}"
            )

            if s>=MIN_UPDATES and bad_evals>=PATIENCE_EVALS:
                stop_reason="CONVERGED_PATIENCE"
                should_stop=True

        # Local checkpoint every 100 steps and at the last/stop step.
        save_ck = (s % 100 == 0) or (s == MAX_UPDATES) or should_stop
        if save_ck:
            torch.cuda.synchronize()
            elapsed_total = elapsed_before + (time.perf_counter()-session_t0)

            torch.save({
                "step":s,
                "model":portable_state_dict(model),
                "best_model":best_sd,
                "raw_best":raw_best,
                "raw_best_step":raw_best_step,
                "convergence_ref":convergence_ref,
                "bad_evals":bad_evals,
                "optimizer":opt.state_dict(),
                "scheduler":sched.state_dict(),
                "history":history,
                "elapsed_train_seconds":elapsed_total,
                "legacy_elapsed_unknown":legacy_elapsed_unknown,
                "run_id":tag,
                "task":task,
                "seed":seed,
            },latest_path)

            # Cross-Colab checkpoint sync every N steps.
            if (
                HF_ENABLED
                and (
                    s % HF_CHECKPOINT_EVERY == 0
                    or s == MAX_UPDATES
                    or should_stop
                )
            ):
                hf_sync_checkpoint(latest_path,tag,s)

        if should_stop:
            break

    torch.cuda.synchronize()
    elapsed_total = elapsed_before + (time.perf_counter()-session_t0)
    peak=int(torch.cuda.max_memory_allocated())

    actual_updates=history[-1]["step"] if history else start_step
    if best_sd is None:
        raw_best=evaluate_model(model,va,task)["rmse"]
        raw_best_step=actual_updates
        best_sd=portable_state_dict(model)

    load_portable_state_dict(model,best_sd)

    return model,{
        "best_val_rmse":float(raw_best),
        "best_val_step":int(raw_best_step),
        "actual_updates":int(actual_updates),
        "stop_reason":stop_reason,
        "train_gpu_seconds":float(elapsed_total),
        "train_gpu_seconds_prior_session_unknown":bool(legacy_elapsed_unknown),
        "peak_vram_bytes":peak,
        "history":history,
    }

# ------------------------------------------------------------------
# Scan restored progress before starting the 180-run loop.
# ------------------------------------------------------------------
initial_state = scan_local_progress()
print(
    f"[START PROGRESS] completed={initial_state['completed']}/"
    f"{initial_state['total_expected']} | "
    f"latest checkpoints={initial_state['latest_checkpoints']}"
)

FORMAL=[]

for seed in SEEDS:
    for task in TASKS:
        dim,c,S=task_spec(task)
        batch=BATCH_1D if dim==1 else BATCH_2D
        _,va,te=loaders_for(task,batch,seed)

        for name in MODELS:
            run_id=f"publication__{name}__{task}__s{seed}"
            result_path=OUT/f"{run_id}_result.json"
            history_path=OUT/f"{run_id}_history.csv"
            latest_path=CKPT/f"{run_id}_latest.pt"

            # Completed result restored from HF or local disk => skip exactly.
            if result_path.exists():
                try:
                    row=json.loads(result_path.read_text(encoding="utf-8"))
                except Exception:
                    row={}
                if row.get("status")=="COMPLETED":
                    FORMAL.append(row)
                    print("[completed resume/skip]",run_id)
                    continue

            seed_all(seed)
            cfg=MATCH[task][name]["config"]
            m=build_by_config(task,name,cfg)
            lr=LR_SELECTED[name]

            print("\n"+"="*105)
            print(
                run_id,
                "params",count_params(m),
                "lr",lr,
                "checkpoint",latest_path.exists()
            )

            # Record current run in local progress manifest.
            write_progress_manifest(
                current_run=run_id,
                current_step=None,
                event="RUN_START",
            )

            try:
                m,ti=train_to_convergence(
                    m,task,va,lr,seed,run_id,latest_path
                )

                if ti["history"]:
                    pd.DataFrame(ti["history"]).to_csv(history_path,index=False)

                ev=evaluate_model(
                    m,te,task,
                    save_examples=(seed==SEEDS[0]),
                    example_tag=f"{task}__{name}"
                )
                lat=measure_latency(m,te)
                arch_diag=architecture_specific_diagnostics(m,te,name)

                row={
                    "seed":seed,
                    "task":task,
                    "model":name,
                    "status":"COMPLETED",
                    "params":count_params(m),
                    "lr":lr,
                    "best_val_rmse":ti["best_val_rmse"],
                    "best_val_step":ti["best_val_step"],
                    "actual_updates":ti["actual_updates"],
                    "stop_reason":ti["stop_reason"],
                    "train_gpu_seconds":ti["train_gpu_seconds"],
                    "train_gpu_seconds_prior_session_unknown":
                        ti["train_gpu_seconds_prior_session_unknown"],
                    "peak_vram_bytes":ti["peak_vram_bytes"],
                    **ev,**lat,**arch_diag,
                }
                save_json(row,result_path)

            except Exception as e:
                row={
                    "seed":seed,
                    "task":task,
                    "model":name,
                    "status":"FAILED",
                    "params":count_params(m),
                    "lr":lr,
                    "error":repr(e),
                }
                save_json(row,result_path)

                # Preserve any history already stored in the checkpoint.
                if latest_path.exists() and not history_path.exists():
                    try:
                        ck=torch.load(
                            latest_path,
                            map_location="cpu",
                            weights_only=False
                        )
                        if ck.get("history"):
                            pd.DataFrame(ck["history"]).to_csv(
                                history_path,
                                index=False
                            )
                    except Exception:
                        pass

            FORMAL.append(row)
            print(row)

            # ----------------------------------------------------------
            # CRITICAL: sync THIS run immediately, before the next run.
            # ----------------------------------------------------------
            hf_sync_finished_run(
                run_id=run_id,
                result_path=result_path,
                history_path=history_path,
                latest_path=latest_path,
            )

            # Completed run no longer needs a local resume checkpoint.
            if row.get("status")=="COMPLETED":
                latest_path.unlink(missing_ok=True)

            del m
            torch.cuda.empty_cache()
            gc.collect()

# Rebuild final formal table from disk so restored + newly completed runs
# are represented even if notebook traversal order changes.
all_rows=[]
for p in sorted(OUT.glob("publication__*_result.json")):
    try:
        all_rows.append(json.loads(p.read_text(encoding="utf-8")))
    except Exception:
        pass

formal_df=pd.DataFrame(all_rows)
formal_df.to_csv(OUT/"publication_run_results.csv",index=False)
display(formal_df)

required=len(SEEDS)*len(TASKS)*len(MODELS)
completed=int((formal_df.status=="COMPLETED").sum()) if len(formal_df) else 0
print("Publication completion:",completed,"/",required)

write_progress_manifest(
    current_run=None,
    current_step=None,
    event="TRAINING_LOOP_END",
)

if completed!=required:
    raise RuntimeError(
        f"Publication benchmark incomplete ({completed}/{required}). "
        "Restart Colab and rerun from the top: HF progress will be restored, "
        "completed runs skipped, and an unfinished run resumed from its latest "
        "HF checkpoint."
    )

REPORT["formal_runs"]=formal_df.to_dict("records")



# 10. Publication statistics

10 seeds 允许我们正式做 paired nonparametric inference。

本轮输出：

- per-task model mean / std / 95% bootstrap CI；
- task-normalized overall score；
- CIDM Core vs all；
- CIDM Complete vs all；
- Core vs Complete；
- paired Wilcoxon；
- Holm correction；
- paired Cohen \(d_z\)；
- win count；
- convergence step / compute efficiency。

注意：统计检验服务于结果描述，不用于“挑选一个更好看的 CIDM 版本”。


In [ ]:

# ============================================================
# 10. Publication statistics
# ============================================================
from scipy.stats import wilcoxon

df=formal_df[formal_df.status=="COMPLETED"].copy()

def bootstrap_ci(values,n_boot=20000,seed=20260929):
    rng=np.random.default_rng(seed)
    x=np.asarray(values,dtype=float)
    b=np.empty(n_boot,dtype=float)
    for i in range(n_boot):
        idx=rng.integers(0,len(x),len(x))
        b[i]=x[idx].mean()
    return float(np.percentile(b,2.5)),float(np.percentile(b,97.5))

# Per task/model summary.
summary_rows=[]
for (task,model),g in df.groupby(["task","model"]):
    lo,hi=bootstrap_ci(g.rmse.values)
    row={
        "task":task,"model":model,
        "rmse_mean":g.rmse.mean(),
        "rmse_std":g.rmse.std(ddof=1),
        "rmse_ci95_low":lo,
        "rmse_ci95_high":hi,
        "mae_mean":g.mae.mean(),
        "rel_l2_mean":g.relative_l2.mean(),
        "rollout_growth_mean":g.rollout_growth.mean(),
        "actual_updates_mean":g.actual_updates.mean(),
        "best_val_step_mean":g.best_val_step.mean(),
        "train_seconds_mean":g.train_gpu_seconds.mean(),
        "latency_ms_mean":g.latency_ms_median.mean(),
        "peak_vram_mean":g.peak_vram_bytes.mean(),
        "band_fraction_low_mean":g.band_fraction_abs_error_low.mean(),
        "band_fraction_mid_mean":g.band_fraction_abs_error_mid.mean(),
        "band_fraction_high_mean":g.band_fraction_abs_error_high.mean(),
        "logspec_low_mean":g.logspec_rmse_low.mean(),
        "logspec_mid_mean":g.logspec_rmse_mid.mean(),
        "logspec_high_mean":g.logspec_rmse_high.mean(),
    }
    for col in ["latent_energy_drift","local_strength","step_size","soc_strength","ber_strength"]:
        if col in g.columns and g[col].notna().any():
            row[col+"_mean"]=g[col].mean()
    summary_rows.append(row)

summary=pd.DataFrame(summary_rows)
summary.to_csv(OUT/"publication_summary_by_task_model.csv",index=False)
display(summary)

# Task-normalized score.
mins=summary.groupby("task")["rmse_mean"].min().to_dict()
summary["task_normalized_rmse_ratio"]=summary.apply(
    lambda r:r.rmse_mean/mins[r.task],axis=1
)
overall=summary.groupby("model").agg(
    mean_task_normalized_rmse=("task_normalized_rmse_ratio","mean"),
    mean_updates=("actual_updates_mean","mean"),
    mean_train_seconds=("train_seconds_mean","mean"),
    mean_latency_ms=("latency_ms_mean","mean"),
    mean_peak_vram=("peak_vram_mean","mean"),
    mean_logspec_high=("logspec_high_mean","mean"),
).reset_index().sort_values("mean_task_normalized_rmse")
overall.to_csv(OUT/"publication_overall_task_normalized.csv",index=False)
display(overall)

def holm_adjust(pvals):
    p=np.asarray(pvals,dtype=float)
    m=len(p)
    order=np.argsort(p)
    adj=np.empty(m,dtype=float)
    running=0.0
    for rank,idx in enumerate(order):
        val=(m-rank)*p[idx]
        running=max(running,val)
        adj[idx]=min(1.0,running)
    return adj

def paired_stats(reference,comparators):
    rows=[]
    for task in TASKS:
        ref=df[(df.task==task)&(df.model==reference)][["seed","rmse"]].rename(
            columns={"rmse":"rmse_ref"}
        )
        task_rows=[]
        for name in comparators:
            cmp=df[(df.task==task)&(df.model==name)][["seed","rmse"]].rename(
                columns={"rmse":"rmse_cmp"}
            )
            m=ref.merge(cmp,on="seed").sort_values("seed")
            diff=(m.rmse_cmp-m.rmse_ref).to_numpy()

            lo,hi=bootstrap_ci(diff)
            try:
                stat,p=wilcoxon(diff,zero_method="wilcox",alternative="two-sided",method="exact")
                stat=float(stat); p=float(p)
            except Exception:
                stat=np.nan; p=np.nan

            sd=diff.std(ddof=1)
            dz=float(diff.mean()/sd) if sd>0 else np.inf*np.sign(diff.mean())

            task_rows.append({
                "task":task,
                "reference":reference,
                "comparator":name,
                "mean_rmse_difference_cmp_minus_ref":float(diff.mean()),
                "bootstrap95_low":lo,
                "bootstrap95_high":hi,
                "wilcoxon_stat":stat,
                "wilcoxon_p_raw":p,
                "paired_cohen_dz":dz,
                "reference_lower_rmse_seed_count":int((diff>0).sum()),
                "n_seeds":len(diff),
            })

        adj=holm_adjust([r["wilcoxon_p_raw"] for r in task_rows])
        for r,a in zip(task_rows,adj):
            r["wilcoxon_p_holm"]=float(a)
        rows.extend(task_rows)
    return pd.DataFrame(rows)

external=[
    "PDEBench-FNO",
    "PDEBench-U-Net",
    "NeuralOperator-FNO",
    "Transolver",
]

core_pairs=paired_stats(
    "CIDM-Foundation-v1",
    external+["CIDM-Complete"]
)
complete_pairs=paired_stats(
    "CIDM-Complete",
    external+["CIDM-Foundation-v1"]
)
core_pairs.to_csv(OUT/"publication_paired_against_core.csv",index=False)
complete_pairs.to_csv(OUT/"publication_paired_against_complete.csv",index=False)

# Direct Core vs Complete accuracy + compute.
cc=df[df.model.isin(["CIDM-Foundation-v1","CIDM-Complete"])].pivot(
    index=["task","seed"],columns="model",
    values=["rmse","train_gpu_seconds","latency_ms_median","peak_vram_bytes","actual_updates"]
).reset_index()

cc_rows=[]
for _,r in cc.iterrows():
    core=float(r[("rmse","CIDM-Foundation-v1")])
    comp=float(r[("rmse","CIDM-Complete")])
    cc_rows.append({
        "task":r[("task","")],
        "seed":int(r[("seed","")]),
        "rmse_core":core,
        "rmse_complete":comp,
        "complete_vs_core_rmse_pct":100*(comp-core)/core,
        "train_seconds_core":float(r[("train_gpu_seconds","CIDM-Foundation-v1")]),
        "train_seconds_complete":float(r[("train_gpu_seconds","CIDM-Complete")]),
        "latency_core_ms":float(r[("latency_ms_median","CIDM-Foundation-v1")]),
        "latency_complete_ms":float(r[("latency_ms_median","CIDM-Complete")]),
        "vram_core":float(r[("peak_vram_bytes","CIDM-Foundation-v1")]),
        "vram_complete":float(r[("peak_vram_bytes","CIDM-Complete")]),
        "updates_core":float(r[("actual_updates","CIDM-Foundation-v1")]),
        "updates_complete":float(r[("actual_updates","CIDM-Complete")]),
    })

cc_df=pd.DataFrame(cc_rows)
cc_df.to_csv(OUT/"publication_Core_vs_Complete_seedwise.csv",index=False)

display(core_pairs)
display(complete_pairs)
display(overall)

REPORT["summary_by_task_model"]=summary.to_dict("records")
REPORT["overall_normalized"]=overall.to_dict("records")
REPORT["paired_against_core"]=core_pairs.to_dict("records")
REPORT["paired_against_complete"]=complete_pairs.to_dict("records")



# 11. Publication-grade figures

绘图不追求“炫”，而追求：
- journal reproducibility；
- exact/prediction/error 一眼可读；
- 黑白打印仍可区分；
- 统一字号与线宽；
- 矢量 PDF/SVG；
- 600 dpi PNG；
- 每张图对应 CSV 数据。

配色只用于模型身份和 signed field，不使用渐变卡片、阴影、3D 柱状图。


In [ ]:
# ============================================================
# 11. Publication-grade figures
# ============================================================
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Journal-like global style.
plt.rcParams.update({
    "figure.facecolor":"white",
    "axes.facecolor":"white",
    "savefig.facecolor":"white",
    "font.family":"serif",
    "font.serif":["Times New Roman","Nimbus Roman","DejaVu Serif"],
    "mathtext.fontset":"stix",
    "font.size":9,
    "axes.titlesize":9,
    "axes.labelsize":9,
    "xtick.labelsize":8,
    "ytick.labelsize":8,
    "legend.fontsize":7.5,
    "axes.linewidth":0.8,
    "xtick.direction":"in",
    "ytick.direction":"in",
    "xtick.major.width":0.7,
    "ytick.major.width":0.7,
    "xtick.minor.width":0.5,
    "ytick.minor.width":0.5,
    "legend.frameon":False,
    "pdf.fonttype":42,
    "ps.fonttype":42,
})

MODEL_ORDER=[
    "CIDM-Foundation-v1",
    "CIDM-Complete",
    "PDEBench-FNO",
    "NeuralOperator-FNO",
    "PDEBench-U-Net",
    "Transolver",
]
MODEL_LABEL={
    "CIDM-Foundation-v1":"CIDM-Core",
    "CIDM-Complete":"CIDM-Complete",
    "PDEBench-FNO":"PDEBench FNO",
    "NeuralOperator-FNO":"NeuralOperator FNO",
    "PDEBench-U-Net":"U-Net",
    "Transolver":"Transolver",
}
# Fixed restrained palette for reproducible paper identity.
MODEL_COLOR={
    "CIDM-Foundation-v1":"#1f4e79",
    "CIDM-Complete":"#b2182b",
    "PDEBench-FNO":"#4d4d4d",
    "NeuralOperator-FNO":"#2b8cbe",
    "PDEBench-U-Net":"#7f7f7f",
    "Transolver":"#6a51a3",
}

def savefig_all(fig,stem):
    for ext in ["pdf","svg","png"]:
        kw={"bbox_inches":"tight"}
        if ext=="png":
            kw["dpi"]=600
        fig.savefig(FIG/f"{stem}.{ext}",**kw)
    plt.close(fig)

# ------------------------------------------------------------
# Fig01: convergence curves, one figure per task
# ------------------------------------------------------------
for task in TASKS:
    fig,ax=plt.subplots(figsize=(3.45,2.55))
    figdata=[]
    for model in MODEL_ORDER:
        histories=[]
        for seed in SEEDS:
            hp=OUT/f"publication__{model}__{task}__s{seed}_history.csv"
            q=pd.read_csv(hp)[["step","val_rmse_original"]].copy()
            q["seed"]=seed
            histories.append(q)
        hh=pd.concat(histories,ignore_index=True)

        # Interpolate only on evaluation-step union within each run's observed range.
        steps=sorted(hh.step.unique())
        rows=[]
        for s in steps:
            vals=[]
            for seed in SEEDS:
                g=hh[hh.seed==seed]
                if s in set(g.step):
                    vals.append(float(g.loc[g.step==s,"val_rmse_original"].iloc[-1]))
            if vals:
                rows.append((s,np.mean(vals),np.std(vals,ddof=1)/np.sqrt(len(vals)),len(vals)))
        q=pd.DataFrame(rows,columns=["step","mean","sem","n"])
        ax.plot(q["step"],q["mean"],lw=1.4,label=MODEL_LABEL[model],color=MODEL_COLOR[model])
        ax.fill_between(q["step"],q["mean"]-1.96*q["sem"],q["mean"]+1.96*q["sem"],
                        alpha=0.10,color=MODEL_COLOR[model],linewidth=0)
        for _,r in q.iterrows():
            figdata.append({
                "task":task,"model":model,"step":int(r["step"]),
                "mean_val_rmse":float(r["mean"]),
                "ci95_low":float(r["mean"]-1.96*r["sem"]),
                "ci95_high":float(r["mean"]+1.96*r["sem"]),
                "n":int(r["n"]),
            })

    ax.set_xlabel("Optimization updates")
    ax.set_ylabel("Validation RMSE")
    ax.set_yscale("log")
    ax.grid(False)
    ax.legend(ncol=2,loc="best")
    ax.text(-0.16,1.04,"(a)",transform=ax.transAxes,fontweight="bold")
    pd.DataFrame(figdata).to_csv(FIGDATA/f"Fig01_convergence_{task}.csv",index=False)
    savefig_all(fig,f"Fig01_convergence_{task}")

# ------------------------------------------------------------
# Fig02: task RMSE means with bootstrap 95% CI
# ------------------------------------------------------------
fig,axes=plt.subplots(1,3,figsize=(7.2,2.45))
figdata=[]
for ax,task in zip(axes,TASKS):
    q=summary[summary.task==task].set_index("model").loc[MODEL_ORDER].reset_index()
    x=np.arange(len(q))
    y=q.rmse_mean.values
    lo=y-q.rmse_ci95_low.values
    hi=q.rmse_ci95_high.values-y
    colors=[MODEL_COLOR[m] for m in q.model]

    ax.bar(x,y,color=colors,width=0.72,edgecolor="black",linewidth=0.4)
    ax.errorbar(x,y,yerr=np.vstack([lo,hi]),fmt="none",ecolor="black",
                capsize=2,elinewidth=0.7,capthick=0.7)
    ax.set_title(task.replace("_"," "))
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABEL[m] for m in q.model],rotation=55,ha="right")
    ax.set_ylabel("RMSE")
    ax.text(-0.15,1.04,f"({chr(97+list(TASKS).index(task))})",
            transform=ax.transAxes,fontweight="bold")

    for _,r in q.iterrows():
        figdata.append(r.to_dict())

fig.tight_layout(w_pad=1.1)
pd.DataFrame(figdata).to_csv(FIGDATA/"Fig02_rmse_comparison.csv",index=False)
savefig_all(fig,"Fig02_rmse_comparison")

# ------------------------------------------------------------
# Fig03: accuracy-compute tradeoff
# x = mean latency; y = task-normalized RMSE
# ------------------------------------------------------------
fig,ax=plt.subplots(figsize=(3.6,2.8))
merged=overall.copy()
for _,r in merged.iterrows():
    m=r["model"]
    ax.scatter(r.mean_latency_ms,r.mean_task_normalized_rmse,
               s=42,color=MODEL_COLOR[m],edgecolor="black",linewidth=0.45,zorder=3)
    ax.annotate(MODEL_LABEL[m],(r.mean_latency_ms,r.mean_task_normalized_rmse),
                xytext=(4,3),textcoords="offset points",fontsize=7)
ax.set_xlabel("Median 6-step latency (ms)")
ax.set_ylabel("Task-normalized RMSE")
ax.grid(False)
pd.DataFrame(merged).to_csv(FIGDATA/"Fig03_core_complete_tradeoff.csv",index=False)
savefig_all(fig,"Fig03_core_complete_tradeoff")

# ------------------------------------------------------------
# Fig04: spectral fidelity (high band log-spectrum RMSE)
# ------------------------------------------------------------
fig,ax=plt.subplots(figsize=(5.8,2.8))
rows=[]
w=0.12
tasks=list(TASKS)
x=np.arange(len(tasks))
for j,m in enumerate(MODEL_ORDER):
    vals=[]
    for task in tasks:
        v=float(summary[(summary.task==task)&(summary.model==m)].logspec_high_mean.iloc[0])
        vals.append(v)
        rows.append({"task":task,"model":m,"logspec_high_rmse":v})
    ax.bar(x+(j-(len(MODEL_ORDER)-1)/2)*w,vals,width=w,
           label=MODEL_LABEL[m],color=MODEL_COLOR[m],
           edgecolor="black",linewidth=0.35)
ax.set_xticks(x)
ax.set_xticklabels([t.replace("_"," ") for t in tasks])
ax.set_ylabel("High-band log-spectrum RMSE")
ax.legend(ncol=3,loc="upper center",bbox_to_anchor=(0.5,1.20))
pd.DataFrame(rows).to_csv(FIGDATA/"Fig04_spectral_fidelity.csv",index=False)
savefig_all(fig,"Fig04_spectral_fidelity")

# ------------------------------------------------------------
# Prediction figures: fixed seed=17 deterministic example.
# 1D: x-t heatmap + line slices
# 2D: target / prediction / |error| at leads 1,3,6
# We show CIDM-Core, CIDM-Complete and strongest external model by task RMSE.
# ------------------------------------------------------------
def load_example(task,model):
    return torch.load(PRED/f"{task}__{model}.pt",map_location="cpu",weights_only=False)

best_external={}
for task in TASKS:
    q=summary[(summary.task==task)&(~summary.model.str.startswith("CIDM"))]
    best_external[task]=q.sort_values("rmse_mean").iloc[0].model

for task in TASKS:
    compare=["CIDM-Foundation-v1","CIDM-Complete",best_external[task]]
    ex0=load_example(task,compare[0])
    target=ex0["target"]

    if target.ndim==3:  # [K,C,X]
        c=0
        fig,axes=plt.subplots(2,2,figsize=(6.8,4.7))
        # True x-time heatmap.
        true=target[:,c].numpy()
        im=axes[0,0].imshow(true,aspect="auto",origin="lower",cmap="RdBu_r")
        axes[0,0].set_title("Exact")
        axes[0,0].set_xlabel("Spatial index")
        axes[0,0].set_ylabel("Lead step")
        fig.colorbar(im,ax=axes[0,0],fraction=0.046,pad=0.03)

        # Complete x-time heatmap.
        comp=load_example(task,"CIDM-Complete")["pred"][:,c].numpy()
        v=max(abs(true).max(),abs(comp).max())
        axes[0,0].images[0].set_clim(-v,v)
        im2=axes[0,1].imshow(comp,aspect="auto",origin="lower",cmap="RdBu_r",vmin=-v,vmax=v)
        axes[0,1].set_title("CIDM-Complete")
        axes[0,1].set_xlabel("Spatial index")
        axes[0,1].set_ylabel("Lead step")
        fig.colorbar(im2,ax=axes[0,1],fraction=0.046,pad=0.03)

        xx=np.arange(true.shape[-1])
        lead_indices=[0,2,5]
        for li in lead_indices:
            axes[1,0].plot(xx,true[li],color="black",lw=1.2,
                           label=f"Exact t+{li+1}" if li==0 else None)
            axes[1,0].plot(xx,comp[li],ls="--",lw=1.0,
                           color=MODEL_COLOR["CIDM-Complete"],
                           label=f"Complete" if li==0 else None)
        axes[1,0].set_xlabel("Spatial index")
        axes[1,0].set_ylabel("Field value")
        axes[1,0].legend()

        for m in compare:
            pr=load_example(task,m)["pred"][:,c].numpy()
            err=np.sqrt(np.mean((pr-true)**2,axis=1))
            axes[1,1].plot(np.arange(1,HORIZON+1),err,marker="o",ms=2.8,lw=1.1,
                           color=MODEL_COLOR[m],label=MODEL_LABEL[m])
        axes[1,1].set_xlabel("Lead step")
        axes[1,1].set_ylabel("Example RMSE")
        axes[1,1].xaxis.set_major_locator(MaxNLocator(integer=True))
        axes[1,1].legend()
        fig.tight_layout()
        savefig_all(fig,"Fig05_prediction_Burgers")

    else:  # [K,C,H,W]
        c=0
        leads=[0,2,5]
        models=["CIDM-Complete",best_external[task]]
        fig,axes=plt.subplots(len(leads),4,figsize=(7.4,5.8))
        for i,li in enumerate(leads):
            exact=target[li,c].numpy()
            p1=load_example(task,models[0])["pred"][li,c].numpy()
            p2=load_example(task,models[1])["pred"][li,c].numpy()
            vmax=max(abs(exact).max(),abs(p1).max(),abs(p2).max())
            for j,(arr,title) in enumerate([
                (exact,"Exact"),
                (p1,"CIDM-Complete"),
                (p2,MODEL_LABEL[models[1]]),
            ]):
                im=axes[i,j].imshow(arr,cmap="RdBu_r",vmin=-vmax,vmax=vmax,origin="lower")
                if i==0: axes[i,j].set_title(title)
                axes[i,j].set_xticks([]); axes[i,j].set_yticks([])
            err=np.abs(p1-exact)
            imerr=axes[i,3].imshow(err,cmap="magma",origin="lower")
            if i==0: axes[i,3].set_title("|Complete − Exact|")
            axes[i,3].set_xticks([]); axes[i,3].set_yticks([])
            axes[i,0].set_ylabel(f"t+{li+1}")
        fig.tight_layout()
        stem="Fig06_prediction_DiffReact" if task=="diff_react_2d" else "Fig07_prediction_SWE"
        savefig_all(fig,stem)

print("Publication figures written to:",FIG)



# 12. Publication confirmation gate

Gate 只检查：
- frozen code / official commits；
- frozen parameter configs；
- 18/18 preflight；
- frozen global LR contract；
- 180/180 convergence runs；
- 10 seeds；
- repaired spectral metrics；
- paired statistics + Holm correction；
- publication figures PDF/SVG/PNG 全部生成。

不要求 CIDM 获胜。


In [ ]:

# ============================================================
# 12. Final audit / gate
# ============================================================
required_runs=len(SEEDS)*len(TASKS)*len(MODELS)

required_fig_stems=[
    "Fig01_convergence_burgers",
    "Fig01_convergence_diff_react_2d",
    "Fig01_convergence_swe_2d",
    "Fig02_rmse_comparison",
    "Fig03_core_complete_tradeoff",
    "Fig04_spectral_fidelity",
    "Fig05_prediction_Burgers",
    "Fig06_prediction_DiffReact",
    "Fig07_prediction_SWE",
]
fig_ok=all(
    all((FIG/f"{stem}.{ext}").exists() for ext in ["pdf","svg","png"])
    for stem in required_fig_stems
)

checks={
    "frozen_cidm_core_exact":REPORT["cidm_freeze"]["exact"],
    "frozen_cidm_complete_exact":REPORT["cidm_complete_freeze"]["exact"],
    "official_commits_exact":all(x["exact"] for x in checkout),
    "parameter_match_frozen_all":bool(match_df["ok"].all()),
    "preflight_18of18":bool(pre_df["pass"].all()) and len(pre_df)==len(TASKS)*len(MODELS),
    "frozen_global_lr_6of6":len(LR_SELECTED)==len(MODELS),
    "ten_seed_protocol":len(SEEDS)==10,
    "formal_180of180":len(df)==required_runs,
    "spectral_metrics_repaired":all(
        c in df.columns for c in [
            "band_fraction_abs_error_low",
            "band_fraction_abs_error_mid",
            "band_fraction_abs_error_high",
            "logspec_rmse_low",
            "logspec_rmse_mid",
            "logspec_rmse_high",
        ]
    ),
    "paired_core_stats_written":(OUT/"publication_paired_against_core.csv").exists(),
    "paired_complete_stats_written":(OUT/"publication_paired_against_complete.csv").exists(),
    "publication_figures_complete":fig_ok,
}

verdict=(
    "PASS_A1_R2_R1_PUBLICATION_CONFIRMATION"
    if all(checks.values())
    else "A1_R2_R1_INCOMPLETE"
)

REPORT["gate"]=checks
REPORT["verdict"]=verdict
REPORT["finished_at"]=time.strftime("%Y-%m-%d %H:%M:%S")
save_json(REPORT,OUT/"CIDM_Foundation_A1_R2_R1_REPORT.json")

lines=[
    "CIDM Foundation A1-R2-R1",
    "Publication-Grade Convergence & Statistical Confirmation",
    "="*88,
    f"VERDICT: {verdict}",
    "",
]
for k,v in checks.items():
    lines.append(f"[{'PASS' if v else 'FAIL'}] {k}")
lines += [
    "",
    "This gate certifies convergence/statistical/figure completeness.",
    "It does not require CIDM-Core or CIDM-Complete to win.",
    "",
    "Next if PASS:",
    "Proceed to A2 Broad PDE Benchmark without modifying frozen CIDM architectures."
]

gate_path=OUT/"A1_R2_R1_gate_report.txt"
gate_path.write_text("\n".join(lines),encoding="utf-8")
print(gate_path.read_text(encoding="utf-8"))



# 13. Package / automatic download / optional HF upload

结果包包含：
- frozen Core / Complete contract；
- official provenance；
- parameter audit；
- 180 run results；
- convergence histories；
- repaired spectral metrics；
- 10-seed statistics；
- publication figure data；
- PDF/SVG/PNG figures；
- deterministic prediction examples；
- gate report。

不包含大型 source repo、raw PDEBench 数据和中间 latest checkpoint。


In [ ]:

# ============================================================
# 13. Package + final HF synchronization + auto-download
# ============================================================
zip_path=OUT/"CIDM_Foundation_A1_R2_R1_PUBLICATION_RESULTS.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if not p.is_file() or p==zip_path:
            continue
        if "sources" in p.parts or "checkpoints" in p.parts:
            continue
        zf.write(p,p.relative_to(OUT))

sha=sha256_file(zip_path)
(OUT/"CIDM_Foundation_A1_R2_R1_PUBLICATION_RESULTS.zip.sha256").write_text(
    sha+"\n",encoding="utf-8"
)
print("ZIP:",zip_path)
print("SHA256:",sha)

# Final repository snapshot. Per-run progress has already been committed
# incrementally; this final upload adds figures/statistics/package.
if HF_ENABLED:
    try:
        final_manifest,_ = write_progress_manifest(
            current_run=None,
            current_step=None,
            event="FINAL_PACKAGE",
        )

        # Upload the completed result tree under the same stable namespace.
        HF_API.upload_folder(
            repo_id=HF_DATASET_REPO_ID,
            repo_type="dataset",
            folder_path=str(OUT),
            path_in_repo=HF_PROGRESS_PREFIX,
            ignore_patterns=[
                "sources/*",
                "checkpoints/*",
            ],
            commit_message="[A1-R2-R1] final publication-confirmation package",
        )
        print(
            f"[HF FINAL PASS] https://huggingface.co/datasets/"
            f"{HF_DATASET_REPO_ID}/tree/main/{HF_PROGRESS_PREFIX}"
        )
    except Exception as e:
        print(f"[HF FINAL WARNING] {e}")
else:
    print("[HF] final upload skipped because HF service is disabled.")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Auto-download unavailable outside Colab:",repr(e))
    print("Result:",zip_path.resolve())
